# Notebook 06 - Ocean Currents

## Purpose

This notebook introduces the first explicit **environmental-flow model** into HydroLab-3D by modeling ocean current as a world-frame water-velocity field and coupling that field to the already validated reduced translational vehicle dynamics.

The central change is physical rather than merely kinematic:

> Hydrodynamic drag must depend on the vehicle's velocity **relative to the surrounding water**, not simply on its velocity relative to the world frame.

Notebook 06 therefore extends the validated HydroLab-3D translational force balance from

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

to

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D(\mathbf{v}-\mathbf{v}_c)
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

where $\mathbf{v}_c$ is the local ocean-current velocity expressed in the HydroLab-3D world frame.

The objective is to build the **smallest trustworthy environmental-flow layer** needed before the complete 6-DOF UUV model is introduced in Notebook 07.

---

## Why This Matters

An underwater vehicle moves through water, not through an inertial vacuum.

The hydrodynamic force experienced by the vehicle depends on its motion relative to the surrounding fluid. A vehicle traveling at $2~\mathrm{m/s}$ through stationary water and a vehicle traveling at $2~\mathrm{m/s}$ while embedded in a $2~\mathrm{m/s}$ following current do not experience the same drag.

The relevant quantity is therefore the **water-relative velocity**:

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

This distinction is essential for physically meaningful simulation of:

- passive drift;
- thrust against or with a current;
- station keeping;
- trajectory tracking;
- disturbance rejection;
- navigation under environmental flow;
- later closed-loop control;
- multi-UUV autonomy;
- and eventual sim-to-real studies.

HydroLab-3D follows an algorithm-first selective-fidelity philosophy: model the physical effects that materially influence the autonomy problem while avoiding unnecessary computational complexity. A compact current-velocity field satisfies that goal without introducing CFD or oceanographic simulation. :chatgpt-content-reference{index="0"}

---

## What We Will Implement

Notebook 06 will introduce and validate a minimal current model consisting of:

1. **World-frame ocean-current velocity**

$$
\mathbf{v}_c
=
\begin{bmatrix}
v_{c,x}\\
v_{c,y}\\
v_{c,z}
\end{bmatrix}
$$

2. **Uniform three-dimensional current**

$$
\mathbf{v}_c(x,y,z,t)
=
\mathbf{v}_{c,0}
$$

3. **Water-relative vehicle velocity**

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

4. **Current-aware quadratic drag**

$$
\mathbf{F}_D
=
-C_D
\left\|
\mathbf{v}_{rel}
\right\|
\mathbf{v}_{rel}
$$

5. Integration of current-aware drag with the already validated:

- thruster force;
- gravity;
- buoyancy;
- force-to-acceleration mapping;
- velocity integration;
- position integration.

6. Physical and analytical validation using:

- zero-current recovery;
- passive drift;
- following current;
- opposing current;
- axis-aligned cases;
- arbitrary 3D current;
- finite-state checks;
- analytical terminal-velocity predictions.

The implementation will remain intentionally compact. Time-varying, depth-dependent, spatially varying, or vortex-like currents will only be introduced later if they materially strengthen the simulator architecture.

---

## What We Inherit From Notebooks 00-05

Notebook 06 does **not** reopen foundational physics that has already been validated.

The development lineage entering this notebook is:

### Notebook 00 - Basic 3D Kinematics

Validated deterministic world-frame position propagation:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

Production capability:

`integrate_position()`

---

### Notebook 01 - Orientation and Reference Frames

Validated HydroLab-3D body-to-world transformation:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi)
$$

Production capabilities include:

- `rotation_x()`
- `rotation_y()`
- `rotation_z()`
- `rotation_body_to_world()`

---

### Notebook 02 - Basic Dynamics

Validated force-to-acceleration conversion:

$$
\mathbf{a}
=
\frac{\mathbf{F}}{m}
$$

and explicit-Euler velocity propagation:

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

Production capabilities:

- `acceleration_from_force()`
- `integrate_velocity()`

---

### Notebook 03 - Hydrodynamic Drag

Validated quadratic drag:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

Production capability:

`quadratic_drag_force()`

Notebook 06 will preserve this validated drag law and, where composition permits, supply it with the physically appropriate **water-relative velocity** rather than rewriting its internal physics.

---

### Notebook 04 - Gravity and Buoyancy

Validated world-frame gravity:

$$
\mathbf{F}_G
=
\begin{bmatrix}
0\\
0\\
-mg
\end{bmatrix}
$$

and buoyancy:

$$
\mathbf{F}_B
=
\begin{bmatrix}
0\\
0\\
\rho gV
\end{bmatrix}
$$

Production capabilities:

- `gravity_force()`
- `buoyancy_force()`

---

### Notebook 05 - Thruster Model

Validated normalized command saturation:

$$
u_{sat}
=
\operatorname{clip}(u,-1,1)
$$

thrust magnitude:

$$
F_T
=
u_{sat}F_{max}
$$

body-frame thrust:

$$
\mathbf{F}_{T,B}
=
F_T\hat{\mathbf{d}}_B
$$

and world-frame transformation:

$$
\mathbf{F}_{T,W}
=
R_B^W
\mathbf{F}_{T,B}
$$

Production capabilities:

- `thruster_force_body()`
- `thruster_force_world()`

The complete HydroLab-3D regression baseline entering Notebook 06 is **69 passing tests**. :chatgpt-content-reference{index="1"}

---

## Mathematical Model

Let the vehicle world-frame translational state contain position

$$
\mathbf{p}
=
\begin{bmatrix}
x\\
y\\
z
\end{bmatrix}
$$

and world-frame velocity

$$
\mathbf{v}
=
\begin{bmatrix}
v_x\\
v_y\\
v_z
\end{bmatrix}
$$

The environment provides an ocean-current velocity

$$
\mathbf{v}_c
=
\begin{bmatrix}
v_{c,x}\\
v_{c,y}\\
v_{c,z}
\end{bmatrix}
$$

at the vehicle state.

The vehicle's velocity relative to the surrounding water is

$$
\boxed{
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
}
$$

The existing quadratic drag model is then evaluated using this relative velocity:

$$
\boxed{
\mathbf{F}_D
=
-C_D
\left\|
\mathbf{v}-\mathbf{v}_c
\right\|
\left(
\mathbf{v}-\mathbf{v}_c
\right)
}
$$

The reduced translational force balance becomes

$$
\boxed{
m\dot{\mathbf{v}}
=
\mathbf{F}_T
-
C_D
\left\|
\mathbf{v}-\mathbf{v}_c
\right\|
\left(
\mathbf{v}-\mathbf{v}_c
\right)
+
\mathbf{F}_G
+
\mathbf{F}_B
}
$$

and therefore

$$
\dot{\mathbf{v}}
=
\frac{
\mathbf{F}_T
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
}{m}
$$

The current itself is **not independently added to position propagation**.

Position continues to evolve from the vehicle's world-frame velocity:

$$
\dot{\mathbf{p}}
=
\mathbf{v}
$$

The environmental coupling occurs through the hydrodynamic interaction produced by $\mathbf{v}_{rel}$.

---

## Ocean-Current Convention

Ocean-current velocity represents the velocity of the surrounding water relative to the HydroLab-3D world frame.

For the initial model:

$$
\mathbf{v}_c(x,y,z,t)
=
\mathbf{v}_{c,0}
$$

so the current is spatially uniform and constant in time.

For example,

$$
\mathbf{v}_c
=
\begin{bmatrix}
0.5\\
0\\
0
\end{bmatrix}
\mathrm{m/s}
$$

means that the water moves in the world $+x$ direction at $0.5~\mathrm{m/s}$.

Similarly,

$$
\mathbf{v}_c
=
\begin{bmatrix}
-0.5\\
0\\
0
\end{bmatrix}
\mathrm{m/s}
$$

represents an equal current in the world $-x$ direction.

The current model belongs conceptually to the **world/environment layer**, consistent with the intended HydroLab-3D architecture in which currents are environmental state rather than vehicle-actuator behavior. :chatgpt-content-reference{index="2"}

---

## Relative-Velocity Convention

Throughout this notebook:

$$
\boxed{
\mathbf{v}_{rel}
=
\mathbf{v}_{vehicle,world}
-
\mathbf{v}_{current,world}
}
$$

This sign convention produces the required physical behavior.

For a stationary vehicle in a positive current:

$$
\mathbf{v}
=
\mathbf{0}
$$

so

$$
\mathbf{v}_{rel}
=
-\mathbf{v}_c
$$

and therefore

$$
\mathbf{F}_D
=
-C_D
\|
-\mathbf{v}_c
\|
(-\mathbf{v}_c)
$$

which produces drag force **in the direction of the current**.

For a vehicle moving exactly with the surrounding water:

$$
\mathbf{v}
=
\mathbf{v}_c
$$

therefore

$$
\mathbf{v}_{rel}
=
\mathbf{0}
$$

and

$$
\mathbf{F}_D
=
\mathbf{0}
$$

as physically required.

---

## Reference-Frame Convention

Notebook 06 retains the existing HydroLab-3D world convention:

- $+x_W$: horizontal world $x$ direction;
- $+y_W$: horizontal world $y$ direction;
- $+z_W$: upward.

Ocean-current vectors introduced in this notebook are expressed in this same world frame.

Vehicle translational velocity used for current coupling is also expressed in the world frame.

Thus the relative-velocity subtraction

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

is performed between quantities represented in the **same reference frame**.

No second orientation convention or current-specific coordinate system is introduced.

Body-frame actuator directions continue to be transformed into the world frame using the already validated

$$
R_B^W
=
R_z(\psi)R_y(\theta)R_x(\phi)
$$

convention before entering the translational force balance.

---

## Numerical Integration Convention

Notebook 06 retains the explicit-Euler integration ordering validated previously.

At timestep $k$, forces are evaluated from the current state:

$$
\mathbf{v}_{rel,k}
=
\mathbf{v}_k
-
\mathbf{v}_{c,k}
$$

$$
\mathbf{F}_{D,k}
=
-C_D
\left\|
\mathbf{v}_{rel,k}
\right\|
\mathbf{v}_{rel,k}
$$

$$
\mathbf{F}_{net,k}
=
\mathbf{F}_{T,k}
+
\mathbf{F}_{D,k}
+
\mathbf{F}_{G,k}
+
\mathbf{F}_{B,k}
$$

$$
\mathbf{a}_k
=
\frac{\mathbf{F}_{net,k}}{m}
$$

Position then uses the beginning-of-step velocity:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

and velocity updates as

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

This preserves numerical continuity with the already validated reduced translational stack rather than introducing an unrelated integration method solely for the current model.

---

## Expected Result

### Passive Drift

Consider a neutrally buoyant vehicle with:

$$
\mathbf{F}_T
=
\mathbf{0}
$$

and initial velocity

$$
\mathbf{v}(0)
=
\mathbf{0}
$$

inside a uniform nonzero current.

Initially,

$$
\mathbf{v}_{rel}(0)
=
-\mathbf{v}_c
$$

so drag accelerates the vehicle in the direction of the surrounding flow.

As the vehicle accelerates,

$$
\mathbf{v}
\rightarrow
\mathbf{v}_c
$$

therefore

$$
\mathbf{v}_{rel}
\rightarrow
\mathbf{0}
$$

and consequently

$$
\mathbf{F}_D
\rightarrow
\mathbf{0}
$$

The asymptotic physical expectation is therefore:

$$
\boxed{
\mathbf{v}
\rightarrow
\mathbf{v}_c
}
$$

This will serve as one of the principal physical validations of Notebook 06.

---

### Thrust in a Uniform Current

For constant positive $x$-axis thrust and an aligned uniform current,

$$
v_{rel,x}
=
v_x-v_{c,x}
$$

At steady state:

$$
F_T
=
C_D
(v_x-v_{c,x})^2
$$

for the corresponding positive relative-flow branch.

Hence the water-relative terminal speed is

$$
\boxed{
v_{rel,\infty}
=
\sqrt{
\frac{F_T}{C_D}
}
}
$$

and the world-frame terminal speed is

$$
\boxed{
v_{x,\infty}
=
v_{c,x}
+
\sqrt{
\frac{F_T}{C_D}
}
}
$$

Using the canonical Notebook 05 values

$$
F_T
=
20~\mathrm{N}
$$

and

$$
C_D
=
5
$$

gives

$$
v_{rel,\infty}
=
2~\mathrm{m/s}
$$

A following current of

$$
v_{c,x}
=
0.5~\mathrm{m/s}
$$

should therefore produce

$$
v_{x,\infty}
=
2.5~\mathrm{m/s}
$$

while an opposing current of

$$
v_{c,x}
=
-0.5~\mathrm{m/s}
$$

should produce

$$
v_{x,\infty}
=
1.5~\mathrm{m/s}
$$

The important invariant is that the terminal **water-relative** speed remains governed by the thrust-drag equilibrium while the **ground-relative** speed shifts with the current.

---

## Validation Criteria

Notebook 06 will be considered successful when the smallest sufficient set of experiments demonstrates that:

1. A zero-current model returns exactly

$$
\mathbf{v}_c
=
\mathbf{0}
$$

2. A configured uniform current returns the intended finite three-vector exactly.

3. Malformed or non-finite current vectors are rejected rather than silently reshaped or propagated.

4. Relative velocity satisfies

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

for known analytical cases.

5. When

$$
\mathbf{v}
=
\mathbf{v}_c
$$

the model produces

$$
\mathbf{v}_{rel}
=
\mathbf{0}
$$

and

$$
\mathbf{F}_D
=
\mathbf{0}
$$

6. A stationary vehicle inside a nonzero current initially experiences hydrodynamic force in the direction of the current.

7. A passive neutrally buoyant vehicle asymptotically approaches the uniform-current velocity:

$$
\mathbf{v}
\rightarrow
\mathbf{v}_c
$$

8. Zero current reproduces the already validated no-current translational behavior.

9. Following and opposing currents shift world-frame terminal velocity consistently with the analytical equilibrium while preserving the expected terminal water-relative speed.

10. Axis-aligned tests exhibit no unintended cross-axis motion.

11. An arbitrary finite 3D current produces the expected relative-velocity vector and drag direction.

12. Position, velocity, relative velocity, force, and acceleration histories remain finite.

13. The current model can be promoted cleanly into the HydroLab-3D production package without rewriting already validated vehicle physics.

14. The promoted production implementation reproduces the canonical notebook validation after explicit re-import.

15. Focused current tests pass.

16. The complete repository regression suite passes without breaking the **69-test baseline** inherited from Notebook 05.

---

## Relationship to HydroLab-3D

The validated data flow introduced by Notebook 06 is conceptually:

    vehicle world velocity
              +
       ocean current field
              ↓
      relative water velocity
              ↓
       hydrodynamic drag
              ↓
          net force
              ↓
     vehicle acceleration
              ↓
       state integration

This adds the first environmental-flow coupling to the simulator while preserving HydroLab-3D's broader architecture:

    World / Environment
           │
           ▼
      Current Field
           │
           ▼
    Vehicle Dynamics
           │
           ▼
     Updated State

The README already identifies ocean currents as part of the environmental world model and as a required disturbance for marine autonomy and control research. :chatgpt-content-reference{index="3"}

Notebook 06 therefore forms the bridge between the reduced translational vehicle developed in Notebooks 00-05 and the higher-fidelity marine vehicle model planned for Notebook 07.

---

## Scope Boundary

Notebook 06 **will** model:

- world-frame current velocity;
- zero current;
- uniform 3D current;
- water-relative velocity;
- current-aware quadratic drag;
- passive drift;
- thrust under following and opposing current;
- arbitrary 3D current;
- analytical equilibrium behavior;
- integration with the validated reduced vehicle stack;
- production promotion;
- regression testing;
- compact scientific artifacts.

Notebook 06 will **not** model:

- computational fluid dynamics;
- Navier-Stokes equations;
- turbulence;
- vortex shedding;
- surface waves;
- detailed oceanography;
- terrain interaction;
- sensor models;
- underwater communication;
- multi-UUV coordination;
- rotational hydrodynamics;
- added mass;
- Coriolis or centripetal effects;
- full 6-DOF marine dynamics;
- controller design;
- ROS 2 integration;
- reinforcement learning.

Those phenomena belong to later stages of HydroLab-3D.

The objective here is not to simulate an ocean.

The objective is to establish a trustworthy relationship between **vehicle motion, moving water, and hydrodynamic drag**.

---

## Expected Outputs

Notebook 06 should generate only artifacts that provide useful engineering evidence.

The expected compact artifact set is:

- a canonical CSV containing relevant histories such as:
  - time;
  - position;
  - vehicle world velocity;
  - current velocity;
  - water-relative velocity;
  - thrust;
  - drag;
  - net force;
  - acceleration;

- a velocity plot comparing vehicle velocity with current velocity;

- a compact relative-velocity or passive-drift visualization if it provides distinct validation evidence;

- a force-balance or terminal-velocity visualization if required to demonstrate the analytical current-thrust relationship.

Exact filenames will be determined by the validated implementation rather than assumed prematurely.

---

## Output Directory

All persistent Notebook 06 artifacts will be stored under:

`results/notebooks/06_ocean_currents/`

Subdirectories will be created only when required, consistent with the existing HydroLab-3D notebook-output convention. :chatgpt-content-reference{index="4"}

---

## Engineering Handoff Target

Notebook 06 completes the reduced translational environmental-flow layer.

After this notebook is validated, Notebook 07 may assume that the following foundational capabilities have already been independently tested:

- 3D kinematics;
- body/world orientation transforms;
- force-to-acceleration conversion;
- velocity integration;
- quadratic hydrodynamic drag;
- gravity;
- buoyancy;
- command-to-thrust conversion;
- thrust saturation;
- thruster direction handling;
- body-to-world thrust transformation;
- world-frame current representation;
- water-relative velocity;
- current-aware drag behavior;
- passive current drift.

The validated reduced translational model handed to Notebook 07 will therefore be:

$$
\boxed{
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D(\mathbf{v}-\mathbf{v}_c)
+
\mathbf{F}_G
+
\mathbf{F}_B
}
$$

Notebook 07 can then increase vehicle-dynamics fidelity toward the classical marine formulation:

$$
M\dot{\nu}
+
C(\nu)\nu
+
D(\nu)\nu
+
g(\eta)
=
\tau
+
\tau_{env}
$$

without reopening the foundational validation of kinematics, reference frames, translational dynamics, drag, hydrostatics, thrusters, or ocean-current-relative translational behavior.

The purpose of Notebook 06 is therefore deliberately narrow:

$$
\boxed{
\text{Establish correct vehicle-water relative dynamics before increasing model fidelity.}
}
$$

---

## Step 1 - Prototype the Uniform-Current and Water-Relative Velocity Primitives

The first implementation step should establish the two quantities that all later Notebook 06 physics depend on:

1. a validated world-frame uniform-current velocity;
2. the vehicle velocity relative to the surrounding water.

No vehicle dynamics are integrated yet. No production files are modified yet.

This keeps the first test deliberately narrow: before coupling currents into hydrodynamic drag, we should prove that the environmental velocity and relative-velocity arithmetic are unambiguous and numerically trustworthy.

### Uniform Current

For the first HydroLab-3D current model, the surrounding water moves with a constant world-frame velocity

$$
\mathbf{v}_c
=
\begin{bmatrix}
v_{c,x}\\
v_{c,y}\\
v_{c,z}
\end{bmatrix}
$$

such that

$$
\mathbf{v}_c(x,y,z,t)
=
\mathbf{v}_{c,0}
$$

The current is therefore independent of vehicle position and time in this notebook's initial model.

This is intentionally simpler than the more general velocity-field concept

$$
\mathbf{v}_c(x,y,z,t)
$$

because spatial and temporal current variation are not required to validate the fundamental vehicle-water coupling.

### Water-Relative Velocity

The vehicle world-frame velocity is

$$
\mathbf{v}
=
\begin{bmatrix}
v_x\\
v_y\\
v_z
\end{bmatrix}
$$

and the water-relative velocity is defined as

$$
\boxed{
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
}
$$

This sign convention is essential.

For a stationary vehicle,

$$
\mathbf{v}
=
\mathbf{0}
$$

inside a current

$$
\mathbf{v}_c
\neq
\mathbf{0}
$$

we obtain

$$
\mathbf{v}_{rel}
=
-\mathbf{v}_c
$$

which will later cause the existing quadratic drag law to generate force in the direction of the current.

Conversely, if the vehicle moves exactly with the surrounding water,

$$
\mathbf{v}
=
\mathbf{v}_c
$$

then

$$
\mathbf{v}_{rel}
=
\mathbf{0}
$$

which must later imply zero hydrodynamic drag.

### Implementation Choice

The current model and relative-velocity calculation will first be implemented as **notebook-local prototypes**.

This follows the existing HydroLab-3D development pattern:

$$
\text{prototype}
\rightarrow
\text{validate}
\rightarrow
\text{couple}
\rightarrow
\text{promote to production}
$$

The already validated `quadratic_drag_force()` will not be modified at this stage.

Inspection of Notebook 03 confirms that the existing drag function accepts an arbitrary finite 3-vector named `velocity` and evaluates

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

Therefore, if Notebook 06 validates

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

the existing drag implementation can later be reused compositionally as

`quadratic_drag_force(relative_velocity, drag_coefficient)`

without reopening or duplicating the validated drag physics.

### Validation Performed in This Step

This first prototype will verify:

- zero current returns exactly the zero vector;
- a known arbitrary 3D current is preserved exactly;
- returned current vectors have shape `(3,)`;
- vehicle-minus-current relative velocity is numerically correct;
- a stationary vehicle produces

$$
\mathbf{v}_{rel}
=
-\mathbf{v}_c
$$

- a vehicle moving exactly with the current produces

$$
\mathbf{v}_{rel}
=
\mathbf{0}
$$

- arbitrary 3D vehicle/current combinations are handled correctly;
- malformed vector shapes are rejected;
- `NaN` and infinite values are rejected;
- caller-owned arrays are not silently aliased by the uniform-current result.

### Expected Result

All checks should pass exactly or to machine precision.

A representative case such as

$$
\mathbf{v}
=
\begin{bmatrix}
2.0\\
-1.0\\
0.5
\end{bmatrix}
\mathrm{m/s}
$$

and

$$
\mathbf{v}_c
=
\begin{bmatrix}
0.5\\
0.25\\
-0.1
\end{bmatrix}
\mathrm{m/s}
$$

should produce

$$
\mathbf{v}_{rel}
=
\begin{bmatrix}
1.5\\
-1.25\\
0.6
\end{bmatrix}
\mathrm{m/s}
$$

If this step passes, the next step can safely connect the validated relative velocity to the already validated quadratic-drag implementation.

In [1]:
import numpy as np


# ---------------------------------------------------------------------
# Notebook-local ocean-current prototypes
# ---------------------------------------------------------------------

def uniform_current_velocity(current_velocity):
    """
    Return a validated uniform ocean-current velocity in the
    HydroLab-3D world frame.

    Parameters
    ----------
    current_velocity : array-like
        Uniform world-frame current velocity [vc_x, vc_y, vc_z] in m/s.

    Returns
    -------
    np.ndarray
        Independent finite 3-vector containing the current velocity.
    """
    current_velocity = np.asarray(
        current_velocity,
        dtype=float,
    )

    if current_velocity.shape != (3,):
        raise ValueError(
            "current_velocity must have shape (3,)"
        )

    if not np.all(np.isfinite(current_velocity)):
        raise ValueError(
            "current_velocity must contain only finite values"
        )

    return current_velocity.copy()


def relative_water_velocity(
    vehicle_velocity,
    current_velocity,
):
    """
    Compute vehicle velocity relative to the surrounding water.

    Both inputs are expressed in the HydroLab-3D world frame.

    v_rel = v_vehicle - v_current

    Parameters
    ----------
    vehicle_velocity : array-like
        Vehicle world-frame velocity [vx, vy, vz] in m/s.

    current_velocity : array-like
        Ocean-current world-frame velocity [vc_x, vc_y, vc_z] in m/s.

    Returns
    -------
    np.ndarray
        Vehicle water-relative velocity [vr_x, vr_y, vr_z] in m/s.
    """
    vehicle_velocity = np.asarray(
        vehicle_velocity,
        dtype=float,
    )

    current_velocity = np.asarray(
        current_velocity,
        dtype=float,
    )

    if vehicle_velocity.shape != (3,):
        raise ValueError(
            "vehicle_velocity must have shape (3,)"
        )

    if current_velocity.shape != (3,):
        raise ValueError(
            "current_velocity must have shape (3,)"
        )

    if not np.all(np.isfinite(vehicle_velocity)):
        raise ValueError(
            "vehicle_velocity must contain only finite values"
        )

    if not np.all(np.isfinite(current_velocity)):
        raise ValueError(
            "current_velocity must contain only finite values"
        )

    return vehicle_velocity - current_velocity


# =====================================================================
# VALIDATION 1 - ZERO CURRENT
# =====================================================================

current_zero = uniform_current_velocity(
    np.zeros(3)
)

np.testing.assert_array_equal(
    current_zero,
    np.zeros(3),
)

assert current_zero.shape == (3,)


# =====================================================================
# VALIDATION 2 - KNOWN UNIFORM 3D CURRENT
# =====================================================================

configured_current = np.array(
    [0.5, 0.25, -0.1],
    dtype=float,
)

current_known = uniform_current_velocity(
    configured_current
)

np.testing.assert_array_equal(
    current_known,
    configured_current,
)

assert current_known.shape == (3,)


# Verify that the returned vector is independent of the caller's array.
assert not np.shares_memory(
    current_known,
    configured_current,
)


# =====================================================================
# VALIDATION 3 - KNOWN RELATIVE VELOCITY
# =====================================================================

vehicle_velocity = np.array(
    [2.0, -1.0, 0.5],
    dtype=float,
)

expected_relative_velocity = np.array(
    [1.5, -1.25, 0.6],
    dtype=float,
)

relative_velocity = relative_water_velocity(
    vehicle_velocity,
    current_known,
)

np.testing.assert_allclose(
    relative_velocity,
    expected_relative_velocity,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# VALIDATION 4 - STATIONARY VEHICLE IN MOVING WATER
# =====================================================================

stationary_relative_velocity = relative_water_velocity(
    np.zeros(3),
    current_known,
)

np.testing.assert_allclose(
    stationary_relative_velocity,
    -current_known,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# VALIDATION 5 - VEHICLE MOVING EXACTLY WITH CURRENT
# =====================================================================

comoving_relative_velocity = relative_water_velocity(
    current_known,
    current_known,
)

np.testing.assert_array_equal(
    comoving_relative_velocity,
    np.zeros(3),
)


# =====================================================================
# VALIDATION 6 - ARBITRARY 3D CASE
# =====================================================================

vehicle_3d = np.array(
    [-1.2, 3.4, -0.8],
    dtype=float,
)

current_3d = np.array(
    [0.3, -0.6, 0.2],
    dtype=float,
)

relative_3d = relative_water_velocity(
    vehicle_3d,
    current_3d,
)

expected_relative_3d = (
    vehicle_3d
    - current_3d
)

np.testing.assert_allclose(
    relative_3d,
    expected_relative_3d,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# VALIDATION 7 - INVALID INPUT REJECTION
# =====================================================================

invalid_cases_passed = 0


def expect_value_error(func):
    global invalid_cases_passed

    try:
        func()
    except ValueError:
        invalid_cases_passed += 1
    else:
        raise AssertionError(
            "Expected ValueError was not raised"
        )


# Invalid uniform-current vectors
expect_value_error(
    lambda: uniform_current_velocity(
        np.ones(2)
    )
)

expect_value_error(
    lambda: uniform_current_velocity(
        np.ones(4)
    )
)

expect_value_error(
    lambda: uniform_current_velocity(
        np.array([0.0, np.nan, 0.0])
    )
)

expect_value_error(
    lambda: uniform_current_velocity(
        np.array([0.0, np.inf, 0.0])
    )
)


# Invalid vehicle/current vectors in relative-velocity calculation
expect_value_error(
    lambda: relative_water_velocity(
        np.ones(2),
        np.zeros(3),
    )
)

expect_value_error(
    lambda: relative_water_velocity(
        np.zeros(3),
        np.ones(2),
    )
)

expect_value_error(
    lambda: relative_water_velocity(
        np.array([0.0, np.nan, 0.0]),
        np.zeros(3),
    )
)

expect_value_error(
    lambda: relative_water_velocity(
        np.zeros(3),
        np.array([0.0, np.inf, 0.0]),
    )
)


assert invalid_cases_passed == 8


# =====================================================================
# SUMMARY
# =====================================================================

print("=" * 80)
print("STEP 1 - OCEAN-CURRENT PRIMITIVE VALIDATION")
print("=" * 80)

print(
    f"\nZero current:"
    f"\n  {current_zero} m/s"
)

print(
    f"\nConfigured current:"
    f"\n  {current_known} m/s"
)

print(
    f"\nVehicle velocity:"
    f"\n  {vehicle_velocity} m/s"
)

print(
    f"\nExpected relative velocity:"
    f"\n  {expected_relative_velocity} m/s"
)

print(
    f"\nComputed relative velocity:"
    f"\n  {relative_velocity} m/s"
)

print(
    f"\nStationary vehicle relative velocity:"
    f"\n  {stationary_relative_velocity} m/s"
)

print(
    f"\nComoving vehicle relative velocity:"
    f"\n  {comoving_relative_velocity} m/s"
)

print(
    f"\nInvalid-input cases rejected:"
    f"\n  {invalid_cases_passed}/8"
)

print(
    "\n[PASS] Uniform-current representation and "
    "water-relative velocity are validated."
)

STEP 1 - OCEAN-CURRENT PRIMITIVE VALIDATION

Zero current:
  [0. 0. 0.] m/s

Configured current:
  [ 0.5   0.25 -0.1 ] m/s

Vehicle velocity:
  [ 2.  -1.   0.5] m/s

Expected relative velocity:
  [ 1.5  -1.25  0.6 ] m/s

Computed relative velocity:
  [ 1.5  -1.25  0.6 ] m/s

Stationary vehicle relative velocity:
  [-0.5  -0.25  0.1 ] m/s

Comoving vehicle relative velocity:
  [0. 0. 0.] m/s

Invalid-input cases rejected:
  8/8

[PASS] Uniform-current representation and water-relative velocity are validated.


## Step 2 - Validate Current-Aware Drag Using the Existing Production Drag Law

Step 1 validated the two environmental-flow primitives required by Notebook 06:

$$
\mathbf{v}_c
$$

and

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

The next step is to establish that ocean-current effects can be coupled cleanly to the already validated HydroLab-3D quadratic-drag implementation.

No new drag law is required.

The existing production function from Notebook 03 evaluates

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

for a supplied finite three-dimensional velocity vector.

For ocean-current-aware motion, the physically relevant velocity supplied to that function is simply the water-relative velocity:

$$
\boxed{
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
}
$$

Therefore,

$$
\boxed{
\mathbf{F}_D
=
-C_D
\|\mathbf{v}_{rel}\|
\mathbf{v}_{rel}
}
$$

or equivalently,

$$
\boxed{
\mathbf{F}_D
=
-C_D
\|\mathbf{v}-\mathbf{v}_c\|
(\mathbf{v}-\mathbf{v}_c)
}
$$

### Architectural Choice

Notebook 06 should preserve the existing drag implementation rather than modifying it.

The desired composition is:

```text
vehicle world velocity
        +
world-frame current velocity
        ↓
water-relative velocity
        ↓
existing quadratic_drag_force()
        ↓
hydrodynamic drag
```

This maintains a clean separation of responsibilities:

- the **world/current layer** describes the surrounding water velocity;
- the **relative-velocity calculation** determines vehicle motion through the fluid;
- the existing **drag model** converts that relative motion into hydrodynamic force.

This also preserves the already validated Notebook 03 drag API and avoids duplicating established physics.

### Import Requirement

Because Jupyter may be launched either from the HydroLab-3D repository root, from `notebooks/`, or from another directory beneath the repository, this step first resolves the actual repository root by searching upward for:

`src/hydrolab3d/`

Only after locating that package will the production drag implementation be imported.

This is purely repository-path handling and does not alter the physical model.

### Validation Performed in This Step

The current-aware drag composition will verify:

1. a vehicle moving exactly with the current has

$$
\mathbf{v}_{rel}
=
\mathbf{0}
$$

and therefore

$$
\mathbf{F}_D
=
\mathbf{0}
$$

2. a stationary vehicle in moving water experiences drag in the direction of the current;

3. a known arbitrary 3D case agrees with the analytical equation;

4. drag remains dissipative relative to the surrounding water:

$$
\mathbf{F}_D
\cdot
\mathbf{v}_{rel}
\le 0
$$

5. with zero current,

$$
\mathbf{v}_c
=
\mathbf{0}
$$

the current-aware formulation reduces exactly to the original Notebook 03 drag behavior:

$$
\mathbf{F}_D(\mathbf{v}-\mathbf{0})
=
\mathbf{F}_D(\mathbf{v})
$$

### Expected Result

All numerical comparisons should agree to machine precision.

Most importantly, this step should demonstrate that **current awareness can be added through composition without changing the validated production drag implementation**.

If this step passes, the next step can use this force inside the reduced translational dynamics and test passive current drift.

In [3]:
import sys
from pathlib import Path

import numpy as np


# =====================================================================
# LOCATE HYDROLAB-3D REPOSITORY
# =====================================================================

cwd = Path.cwd().resolve()

search_locations = [
    cwd,
    *cwd.parents,
]

repo_root = None

for candidate in search_locations:
    package_path = (
        candidate
        / "src"
        / "hydrolab3d"
    )

    if package_path.is_dir():
        repo_root = candidate
        break


if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the HydroLab-3D repository root.\n"
        f"Current working directory: {cwd}\n"
        "Expected to find: src/hydrolab3d/\n"
        "Launch this notebook from somewhere inside the "
        "HydroLab-3D repository."
    )


src_path = repo_root / "src"

if not src_path.is_dir():
    raise FileNotFoundError(
        f"HydroLab-3D source directory not found: {src_path}"
    )


src_path_str = str(src_path)

if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)


print("=" * 80)
print("HYDROLAB-3D PRODUCTION PACKAGE RESOLUTION")
print("=" * 80)
print(f"Current working directory : {cwd}")
print(f"Repository root           : {repo_root}")
print(f"Production source path    : {src_path}")


# =====================================================================
# IMPORT VALIDATED PRODUCTION DRAG IMPLEMENTATION
# =====================================================================

from hydrolab3d.dynamics import quadratic_drag_force


drag_module = Path(
    sys.modules[
        quadratic_drag_force.__module__
    ].__file__
).resolve()


expected_drag_module = (
    src_path
    / "hydrolab3d"
    / "dynamics"
    / "drag.py"
).resolve()


assert drag_module == expected_drag_module, (
    "quadratic_drag_force() was not imported from the expected "
    "HydroLab-3D production module.\n"
    f"Expected: {expected_drag_module}\n"
    f"Actual:   {drag_module}"
)


print(
    "quadratic_drag_force  : "
    f"{drag_module.relative_to(repo_root)}"
)

print(
    "[PASS] Validated production drag implementation imported."
)


# =====================================================================
# NOTEBOOK-LOCAL CURRENT-AWARE COMPOSITION
# =====================================================================

def current_aware_drag_force(
    vehicle_velocity,
    current_velocity,
    drag_coefficient,
):
    """
    Compute quadratic hydrodynamic drag using vehicle velocity
    relative to the surrounding water.

    Parameters
    ----------
    vehicle_velocity : array-like
        Vehicle world-frame velocity [vx, vy, vz] in m/s.

    current_velocity : array-like
        Ocean-current world-frame velocity [vc_x, vc_y, vc_z] in m/s.

    drag_coefficient : float
        Effective quadratic drag coefficient.

    Returns
    -------
    drag_force : np.ndarray
        World-frame hydrodynamic drag force in N.

    relative_velocity : np.ndarray
        Vehicle velocity relative to the surrounding water in m/s.
    """
    relative_velocity = relative_water_velocity(
        vehicle_velocity,
        current_velocity,
    )

    drag_force = quadratic_drag_force(
        relative_velocity,
        drag_coefficient,
    )

    return drag_force, relative_velocity


# =====================================================================
# VALIDATION 1 - COMOVING VEHICLE -> ZERO RELATIVE VELOCITY / DRAG
# =====================================================================

drag_coefficient = 3.0

comoving_velocity = np.array(
    [0.5, 0.25, -0.1],
    dtype=float,
)

comoving_current = np.array(
    [0.5, 0.25, -0.1],
    dtype=float,
)


drag_comoving, relative_comoving = current_aware_drag_force(
    vehicle_velocity=comoving_velocity,
    current_velocity=comoving_current,
    drag_coefficient=drag_coefficient,
)


np.testing.assert_array_equal(
    relative_comoving,
    np.zeros(3),
)

np.testing.assert_array_equal(
    drag_comoving,
    np.zeros(3),
)


# =====================================================================
# VALIDATION 2 - STATIONARY VEHICLE IN MOVING WATER
# =====================================================================

stationary_velocity = np.zeros(
    3,
    dtype=float,
)

stationary_current = np.array(
    [0.5, 0.25, -0.1],
    dtype=float,
)


drag_stationary, relative_stationary = current_aware_drag_force(
    vehicle_velocity=stationary_velocity,
    current_velocity=stationary_current,
    drag_coefficient=drag_coefficient,
)


expected_relative_stationary = (
    -stationary_current
)

expected_drag_stationary = (
    -drag_coefficient
    * np.linalg.norm(
        expected_relative_stationary
    )
    * expected_relative_stationary
)


np.testing.assert_allclose(
    relative_stationary,
    expected_relative_stationary,
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    drag_stationary,
    expected_drag_stationary,
    rtol=0.0,
    atol=1e-12,
)


# Drag must accelerate a stationary vehicle
# in the direction of the current.
stationary_alignment = np.dot(
    drag_stationary,
    stationary_current,
)

assert stationary_alignment > 0.0


# =====================================================================
# VALIDATION 3 - KNOWN ARBITRARY 3D CASE
# =====================================================================

vehicle_known = np.array(
    [2.0, -1.0, 0.5],
    dtype=float,
)

current_known = np.array(
    [0.5, 0.25, -0.1],
    dtype=float,
)


drag_known, relative_known = current_aware_drag_force(
    vehicle_velocity=vehicle_known,
    current_velocity=current_known,
    drag_coefficient=drag_coefficient,
)


expected_relative_known = np.array(
    [1.5, -1.25, 0.6],
    dtype=float,
)

expected_drag_known = (
    -drag_coefficient
    * np.linalg.norm(
        expected_relative_known
    )
    * expected_relative_known
)


np.testing.assert_allclose(
    relative_known,
    expected_relative_known,
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    drag_known,
    expected_drag_known,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# VALIDATION 4 - DRAG DISSIPATES WATER-RELATIVE MOTION
# =====================================================================

relative_drag_power = np.dot(
    drag_known,
    relative_known,
)

assert relative_drag_power < 0.0


# =====================================================================
# VALIDATION 5 - ZERO-CURRENT RECOVERY
# =====================================================================

vehicle_zero_current = np.array(
    [1.2, -0.4, 0.9],
    dtype=float,
)

zero_current = np.zeros(
    3,
    dtype=float,
)


drag_current_aware_zero, relative_zero_current = (
    current_aware_drag_force(
        vehicle_velocity=vehicle_zero_current,
        current_velocity=zero_current,
        drag_coefficient=drag_coefficient,
    )
)


drag_original = quadratic_drag_force(
    vehicle_zero_current,
    drag_coefficient,
)


np.testing.assert_allclose(
    relative_zero_current,
    vehicle_zero_current,
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    drag_current_aware_zero,
    drag_original,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# FINITE-VALUE CHECKS
# =====================================================================

arrays_to_check = [
    relative_comoving,
    drag_comoving,
    relative_stationary,
    drag_stationary,
    relative_known,
    drag_known,
    relative_zero_current,
    drag_current_aware_zero,
    drag_original,
]

assert all(
    np.all(np.isfinite(array))
    for array in arrays_to_check
)


# =====================================================================
# SUMMARY
# =====================================================================

print()
print("=" * 80)
print("STEP 2 - CURRENT-AWARE DRAG VALIDATION")
print("=" * 80)


print("\nCase 1 - Vehicle moving exactly with current")
print(
    f"  Vehicle velocity  : "
    f"{comoving_velocity} m/s"
)
print(
    f"  Current velocity  : "
    f"{comoving_current} m/s"
)
print(
    f"  Relative velocity : "
    f"{relative_comoving} m/s"
)
print(
    f"  Drag force        : "
    f"{drag_comoving} N"
)


print("\nCase 2 - Stationary vehicle in moving water")
print(
    f"  Vehicle velocity  : "
    f"{stationary_velocity} m/s"
)
print(
    f"  Current velocity  : "
    f"{stationary_current} m/s"
)
print(
    f"  Relative velocity : "
    f"{relative_stationary} m/s"
)
print(
    f"  Drag force        : "
    f"{drag_stationary} N"
)
print(
    f"  Drag/current dot  : "
    f"{stationary_alignment:.12f}"
)


print("\nCase 3 - Arbitrary 3D motion")
print(
    f"  Vehicle velocity  : "
    f"{vehicle_known} m/s"
)
print(
    f"  Current velocity  : "
    f"{current_known} m/s"
)
print(
    f"  Relative velocity : "
    f"{relative_known} m/s"
)
print(
    f"  Drag force        : "
    f"{drag_known} N"
)


print("\nWater-relative dissipativity")
print(
    f"  F_D · v_rel       : "
    f"{relative_drag_power:.12f}"
)


print("\nZero-current recovery")
print(
    f"  Relative velocity : "
    f"{relative_zero_current} m/s"
)
print(
    f"  Current-aware drag: "
    f"{drag_current_aware_zero} N"
)
print(
    f"  Original drag     : "
    f"{drag_original} N"
)


print(
    "\n[PASS] Current-aware drag is validated using "
    "the existing HydroLab-3D production drag law."
)

HYDROLAB-3D PRODUCTION PACKAGE RESOLUTION
Current working directory : /home/agniacolyte/HydroLab-3D/notebooks
Repository root           : /home/agniacolyte/HydroLab-3D
Production source path    : /home/agniacolyte/HydroLab-3D/src
quadratic_drag_force  : src/hydrolab3d/dynamics/drag.py
[PASS] Validated production drag implementation imported.

STEP 2 - CURRENT-AWARE DRAG VALIDATION

Case 1 - Vehicle moving exactly with current
  Vehicle velocity  : [ 0.5   0.25 -0.1 ] m/s
  Current velocity  : [ 0.5   0.25 -0.1 ] m/s
  Relative velocity : [0. 0. 0.] m/s
  Drag force        : [-0. -0. -0.] N

Case 2 - Stationary vehicle in moving water
  Vehicle velocity  : [0. 0. 0.] m/s
  Current velocity  : [ 0.5   0.25 -0.1 ] m/s
  Relative velocity : [-0.5  -0.25  0.1 ] m/s
  Drag force        : [ 0.85183625  0.42591813 -0.17036725] N
  Drag/current dot  : 0.549434382456

Case 3 - Arbitrary 3D motion
  Vehicle velocity  : [ 2.  -1.   0.5] m/s
  Current velocity  : [ 0.5   0.25 -0.1 ] m/s
  Relative 

## Step 3 - Validate Passive Drift in a Uniform Ocean Current

Steps 1 and 2 established the two pieces required for current-aware hydrodynamics:

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

and

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}_{rel}\|
\mathbf{v}_{rel}
$$

The next step is the first **dynamic validation** of the ocean-current model.

We will place a passive, neutrally buoyant vehicle at rest inside a constant three-dimensional current and verify that hydrodynamic drag accelerates the vehicle toward the surrounding water velocity.

No thruster force is applied.

---

### Physical Setup

For a passive vehicle,

$$
\mathbf{F}_T
=
\mathbf{0}
$$

The full reduced translational force balance is

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

For neutral buoyancy,

$$
\mathbf{F}_G
+
\mathbf{F}_B
=
\mathbf{0}
$$

so the dynamics reduce to

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_D
$$

with

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}-\mathbf{v}_c\|
(\mathbf{v}-\mathbf{v}_c)
$$

The vehicle begins from rest:

$$
\mathbf{v}(0)
=
\mathbf{0}
$$

while the surrounding water has a nonzero uniform velocity:

$$
\mathbf{v}_c
\neq
\mathbf{0}
$$

Initially,

$$
\mathbf{v}_{rel}(0)
=
-\mathbf{v}_c
$$

and therefore drag acts in the same direction as the current.

As the vehicle accelerates,

$$
\|\mathbf{v}-\mathbf{v}_c\|
$$

decreases.

Consequently,

$$
\|\mathbf{F}_D\|
$$

also decreases.

The expected asymptotic behavior is

$$
\boxed{
\mathbf{v}
\rightarrow
\mathbf{v}_c
}
$$

and therefore

$$
\boxed{
\mathbf{v}_{rel}
\rightarrow
\mathbf{0}
}
$$

and

$$
\boxed{
\mathbf{F}_D
\rightarrow
\mathbf{0}
}
$$

This is the defining passive-drift behavior required from the reduced ocean-current model.

---

### Important Modeling Point

The current velocity will **not** be added directly to vehicle position.

Position continues to evolve from the simulated vehicle world velocity:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

The ocean current influences the trajectory only through the force produced by water-relative motion:

    ocean current
         +
    vehicle velocity
         ↓
    relative velocity
         ↓
         drag
         ↓
      acceleration
         ↓
    vehicle velocity
         ↓
       position

This prevents double-counting the environmental flow.

---

### Numerical Integration

The simulation retains the explicit-Euler ordering validated in the earlier HydroLab-3D notebooks.

At each timestep:

$$
\mathbf{v}_{rel,k}
=
\mathbf{v}_k
-
\mathbf{v}_c
$$

$$
\mathbf{F}_{D,k}
=
-C_D
\|\mathbf{v}_{rel,k}\|
\mathbf{v}_{rel,k}
$$

$$
\mathbf{F}_{net,k}
=
\mathbf{F}_{D,k}
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

$$
\mathbf{a}_k
=
\frac{\mathbf{F}_{net,k}}{m}
$$

Position is updated first using the beginning-of-step velocity:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

followed by velocity:

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

No new integration method is introduced.

---

### Canonical Passive-Drift Case

The simulation will use:

$$
m
=
20\text{ kg}
$$

$$
\rho
=
1000\text{ kg/m}^3
$$

$$
V
=
0.020\text{ m}^3
$$

which gives exact neutral buoyancy because

$$
\rho V
=
m
$$

The drag coefficient is

$$
C_D
=
5
$$

and the imposed uniform three-dimensional current is

$$
\mathbf{v}_c
=
\begin{bmatrix}
0.50\\
-0.25\\
0.10
\end{bmatrix}
\text{ m/s}
$$

The vehicle begins from

$$
\mathbf{p}_0
=
\begin{bmatrix}
0\\
0\\
-10
\end{bmatrix}
\text{ m}
$$

and

$$
\mathbf{v}_0
=
\mathbf{0}
$$

---

### Analytical Relative-Speed Reference

Because the current is constant and the only nonzero net force is isotropic quadratic drag, define

$$
\mathbf{u}
=
\mathbf{v}
-
\mathbf{v}_c
$$

Since

$$
\dot{\mathbf{v}}_c
=
\mathbf{0}
$$

we obtain

$$
m\dot{\mathbf{u}}
=
-C_D
\|\mathbf{u}\|
\mathbf{u}
$$

For this passive uniform-current case, the direction of $\mathbf{u}$ remains constant while its magnitude decays.

Let

$$
s(t)
=
\|\mathbf{u}(t)\|
$$

Then

$$
m\dot{s}
=
-C_Ds^2
$$

which gives

$$
\boxed{
s(t)
=
\frac{s_0}{
1+
\frac{C_D}{m}s_0t
}
}
$$

where

$$
s_0
=
\|\mathbf{v}_0-\mathbf{v}_c\|
$$

This provides a stronger validation than checking only qualitative drift.

The numerical water-relative speed should closely follow this analytical decay.

---

### Validation Criteria

This step passes only if:

1. gravity and buoyancy cancel to numerical precision;

2. the initial drag force points in the direction of the current;

3. vehicle speed increases from rest toward the current velocity;

4. water-relative speed decreases monotonically;

5. the final vehicle velocity is substantially closer to

$$
\mathbf{v}_c
$$

than the initial vehicle velocity;

6. the numerical relative-speed history agrees closely with

$$
s(t)
=
\frac{s_0}{
1+
\frac{C_D}{m}s_0t
}
$$

7. drag magnitude decreases as the vehicle approaches the surrounding flow;

8. the final drag magnitude is much smaller than its initial value;

9. the vehicle trajectory develops motion in the physically expected signs of all three current axes;

10. all positions, velocities, accelerations, forces, and relative velocities remain finite.

If these conditions are satisfied, HydroLab-3D will have demonstrated the fundamental physical behavior required from a uniform-current model:

$$
\boxed{
\text{a passive neutrally buoyant vehicle is entrained toward the surrounding flow}
}
$$

In [4]:
from hydrolab3d.dynamics.kinematics import integrate_position
from hydrolab3d.dynamics.rigid_body import (
    acceleration_from_force,
    integrate_velocity,
)
from hydrolab3d.dynamics.buoyancy import (
    gravity_force,
    buoyancy_force,
)


# =====================================================================
# PASSIVE-DRIFT PARAMETERS
# =====================================================================

mass = 20.0
fluid_density = 1000.0
gravity = 9.81

displaced_volume = mass / fluid_density

drag_coefficient = 5.0

dt = 0.01
duration = 120.0

num_steps = int(round(duration / dt))


current_velocity = np.array(
    [0.50, -0.25, 0.10],
    dtype=float,
)

initial_position = np.array(
    [0.0, 0.0, -10.0],
    dtype=float,
)

initial_velocity = np.zeros(
    3,
    dtype=float,
)


# =====================================================================
# VERIFY NEUTRAL BUOYANCY
# =====================================================================

gravity_vector = gravity_force(
    mass,
    gravity,
)

buoyancy_vector = buoyancy_force(
    fluid_density,
    displaced_volume,
    gravity,
)

hydrostatic_force = (
    gravity_vector
    + buoyancy_vector
)


np.testing.assert_allclose(
    hydrostatic_force,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# INITIALIZE SIMULATION
# =====================================================================

position = initial_position.copy()
velocity = initial_velocity.copy()


initial_relative_velocity = relative_water_velocity(
    velocity,
    current_velocity,
)

initial_drag, _ = current_aware_drag_force(
    vehicle_velocity=velocity,
    current_velocity=current_velocity,
    drag_coefficient=drag_coefficient,
)


# Initial drag must point with the current.
assert np.dot(
    initial_drag,
    current_velocity,
) > 0.0


time_history = [0.0]

position_history = [
    position.copy()
]

velocity_history = [
    velocity.copy()
]

current_history = [
    current_velocity.copy()
]

relative_velocity_history = [
    initial_relative_velocity.copy()
]

drag_history = [
    initial_drag.copy()
]

net_force_history = [
    (
        initial_drag
        + hydrostatic_force
    ).copy()
]

acceleration_history = [
    acceleration_from_force(
        initial_drag
        + hydrostatic_force,
        mass,
    )
]


# =====================================================================
# PASSIVE-DRIFT SIMULATION
# =====================================================================

for step in range(num_steps):

    drag_force, relative_velocity = (
        current_aware_drag_force(
            vehicle_velocity=velocity,
            current_velocity=current_velocity,
            drag_coefficient=drag_coefficient,
        )
    )

    net_force = (
        drag_force
        + hydrostatic_force
    )

    acceleration = acceleration_from_force(
        net_force,
        mass,
    )

    # Preserve the validated HydroLab-3D explicit-Euler ordering:
    # position uses the beginning-of-step velocity.
    position = integrate_position(
        position,
        velocity,
        dt,
    )

    velocity = integrate_velocity(
        velocity,
        acceleration,
        dt,
    )

    time_history.append(
        (step + 1) * dt
    )

    position_history.append(
        position.copy()
    )

    velocity_history.append(
        velocity.copy()
    )

    current_history.append(
        current_velocity.copy()
    )

    relative_velocity_history.append(
        relative_water_velocity(
            velocity,
            current_velocity,
        )
    )

    drag_history.append(
        drag_force.copy()
    )

    net_force_history.append(
        net_force.copy()
    )

    acceleration_history.append(
        acceleration.copy()
    )


# =====================================================================
# CONVERT HISTORIES TO ARRAYS
# =====================================================================

time_history = np.asarray(
    time_history,
)

position_history = np.asarray(
    position_history,
)

velocity_history = np.asarray(
    velocity_history,
)

current_history = np.asarray(
    current_history,
)

relative_velocity_history = np.asarray(
    relative_velocity_history,
)

drag_history = np.asarray(
    drag_history,
)

net_force_history = np.asarray(
    net_force_history,
)

acceleration_history = np.asarray(
    acceleration_history,
)


# =====================================================================
# ANALYTICAL WATER-RELATIVE SPEED
# =====================================================================

initial_relative_speed = np.linalg.norm(
    initial_relative_velocity
)

relative_speed_history = np.linalg.norm(
    relative_velocity_history,
    axis=1,
)

analytical_relative_speed = (
    initial_relative_speed
    / (
        1.0
        + (
            drag_coefficient
            / mass
        )
        * initial_relative_speed
        * time_history
    )
)


# =====================================================================
# PHYSICAL VALIDATION
# =====================================================================

final_velocity = velocity_history[-1]
final_relative_velocity = (
    relative_velocity_history[-1]
)

final_relative_speed = (
    relative_speed_history[-1]
)


initial_velocity_error = np.linalg.norm(
    initial_velocity
    - current_velocity
)

final_velocity_error = np.linalg.norm(
    final_velocity
    - current_velocity
)


# Vehicle must move substantially closer to the current velocity.
assert final_velocity_error < initial_velocity_error


# Relative speed should monotonically decrease.
relative_speed_differences = np.diff(
    relative_speed_history
)

assert np.all(
    relative_speed_differences <= 1e-12
)


# Numerical relative-speed decay should agree closely
# with the analytical quadratic-drag solution.
max_relative_speed_error = np.max(
    np.abs(
        relative_speed_history
        - analytical_relative_speed
    )
)

assert max_relative_speed_error < 5e-4


# Final velocity components must move in the same signs
# as the imposed current.
assert final_velocity[0] > 0.0
assert final_velocity[1] < 0.0
assert final_velocity[2] > 0.0


# Position displacement must also follow the current direction.
displacement = (
    position_history[-1]
    - initial_position
)

assert displacement[0] > 0.0
assert displacement[1] < 0.0
assert displacement[2] > 0.0


# Drag magnitude must decrease strongly during entrainment.
drag_magnitude_history = np.linalg.norm(
    drag_history,
    axis=1,
)

assert (
    drag_magnitude_history[-1]
    < drag_magnitude_history[0]
)


# =====================================================================
# FINITE-STATE VALIDATION
# =====================================================================

arrays_to_check = [
    position_history,
    velocity_history,
    current_history,
    relative_velocity_history,
    drag_history,
    net_force_history,
    acceleration_history,
    analytical_relative_speed,
]

assert all(
    np.all(np.isfinite(array))
    for array in arrays_to_check
)


# =====================================================================
# SUMMARY
# =====================================================================

print("=" * 80)
print("STEP 3 - PASSIVE UNIFORM-CURRENT DRIFT VALIDATION")
print("=" * 80)


print("\nPhysical parameters")
print(
    f"  Mass                     : "
    f"{mass:.6f} kg"
)
print(
    f"  Fluid density            : "
    f"{fluid_density:.6f} kg/m^3"
)
print(
    f"  Displaced volume         : "
    f"{displaced_volume:.6f} m^3"
)
print(
    f"  Drag coefficient         : "
    f"{drag_coefficient:.6f}"
)


print("\nHydrostatic equilibrium")
print(
    f"  Gravity force            : "
    f"{gravity_vector} N"
)
print(
    f"  Buoyancy force           : "
    f"{buoyancy_vector} N"
)
print(
    f"  Gravity + buoyancy       : "
    f"{hydrostatic_force} N"
)


print("\nUniform current")
print(
    f"  Current velocity         : "
    f"{current_velocity} m/s"
)


print("\nInitial state")
print(
    f"  Vehicle velocity         : "
    f"{initial_velocity} m/s"
)
print(
    f"  Relative velocity        : "
    f"{initial_relative_velocity} m/s"
)
print(
    f"  Relative speed           : "
    f"{initial_relative_speed:.12f} m/s"
)
print(
    f"  Initial drag             : "
    f"{initial_drag} N"
)


print("\nFinal state")
print(
    f"  Vehicle velocity         : "
    f"{final_velocity} m/s"
)
print(
    f"  Current velocity         : "
    f"{current_velocity} m/s"
)
print(
    f"  Relative velocity        : "
    f"{final_relative_velocity} m/s"
)
print(
    f"  Relative speed           : "
    f"{final_relative_speed:.12f} m/s"
)
print(
    f"  Final drag magnitude     : "
    f"{drag_magnitude_history[-1]:.12f} N"
)
print(
    f"  Final position           : "
    f"{position_history[-1]} m"
)


print("\nAnalytical validation")
print(
    f"  Analytical final "
    f"relative speed             : "
    f"{analytical_relative_speed[-1]:.12f} m/s"
)
print(
    f"  Numerical final "
    f"relative speed             : "
    f"{final_relative_speed:.12f} m/s"
)
print(
    f"  Max relative-speed error : "
    f"{max_relative_speed_error:.12e} m/s"
)


print("\nCurrent entrainment")
print(
    f"  Initial |v - vc|         : "
    f"{initial_velocity_error:.12f} m/s"
)
print(
    f"  Final |v - vc|           : "
    f"{final_velocity_error:.12f} m/s"
)


print(
    "\n[PASS] Passive neutrally buoyant vehicle "
    "is correctly entrained toward the uniform ocean current."
)

STEP 3 - PASSIVE UNIFORM-CURRENT DRIFT VALIDATION

Physical parameters
  Mass                     : 20.000000 kg
  Fluid density            : 1000.000000 kg/m^3
  Displaced volume         : 0.020000 m^3
  Drag coefficient         : 5.000000

Hydrostatic equilibrium
  Gravity force            : [   0.     0.  -196.2] N
  Buoyancy force           : [  0.    0.  196.2] N
  Gravity + buoyancy       : [0. 0. 0.] N

Uniform current
  Current velocity         : [ 0.5  -0.25  0.1 ] m/s

Initial state
  Vehicle velocity         : [0. 0. 0.] m/s
  Relative velocity        : [-0.5   0.25 -0.1 ] m/s
  Relative speed           : 0.567890834580 m/s
  Initial drag             : [ 1.41972709 -0.70986354  0.28394542] N

Final state
  Vehicle velocity         : [ 0.47228509 -0.23614255  0.09445702] m/s
  Current velocity         : [ 0.5  -0.25  0.1 ] m/s
  Relative velocity        : [-0.02771491  0.01385745 -0.00554298] m/s
  Relative speed           : 0.031478083222 m/s
  Final drag magnitude     : 0.0

## Step 4 - Validate Thrust Equilibrium Under Following and Opposing Currents

Step 3 established that a passive neutrally buoyant vehicle is correctly entrained toward a uniform ocean current.

The next validation introduces propulsion.

The purpose of this step is to demonstrate an important distinction between:

- **water-relative terminal velocity**, determined by the thrust-drag equilibrium;
- **world-frame terminal velocity**, shifted by the ocean current.

This is one of the strongest analytical validations available for the reduced current model.

---

### Physical Model

For a neutrally buoyant vehicle with constant thrust,

$$
\mathbf{F}_G
+
\mathbf{F}_B
=
\mathbf{0}
$$

so the reduced translational dynamics become

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D
$$

with current-aware drag

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}-\mathbf{v}_c\|
(\mathbf{v}-\mathbf{v}_c)
$$

For this validation, thrust and current are both restricted to the world $x$ axis.

Therefore,

$$
v_{rel,x}
=
v_x-v_{c,x}
$$

and the scalar equilibrium condition is

$$
F_T
-
C_D
v_{rel,x}^2
=
0
$$

for positive thrust and positive terminal water-relative motion.

Thus,

$$
\boxed{
v_{rel,\infty}
=
\sqrt{
\frac{F_T}{C_D}
}
}
$$

The world-frame terminal velocity is then

$$
\boxed{
v_{x,\infty}
=
v_{c,x}
+
v_{rel,\infty}
}
$$

or

$$
\boxed{
v_{x,\infty}
=
v_{c,x}
+
\sqrt{
\frac{F_T}{C_D}
}
}
$$

---

### Canonical Thruster Configuration

We reuse the validated Notebook 05 configuration:

$$
F_{max}
=
40\text{ N}
$$

with normalized command

$$
u
=
0.5
$$

Therefore,

$$
F_T
=
uF_{max}
=
20\text{ N}
$$

The drag coefficient is

$$
C_D
=
5
$$

so the expected water-relative terminal speed is

$$
v_{rel,\infty}
=
\sqrt{
\frac{20}{5}
}
=
2.0\text{ m/s}
$$

This value must remain the same regardless of a constant uniform current.

---

### Three Canonical Current Cases

We will compare three otherwise identical simulations.

#### Case A - No Current

$$
v_{c,x}
=
0
$$

Therefore,

$$
v_{x,\infty}
=
2.0\text{ m/s}
$$

---

#### Case B - Following Current

$$
v_{c,x}
=
0.5\text{ m/s}
$$

The current moves in the same direction as the vehicle thrust.

The expected world-frame terminal velocity is

$$
v_{x,\infty}
=
0.5+2.0
=
2.5\text{ m/s}
$$

but the expected terminal water-relative speed remains

$$
v_{rel,\infty}
=
2.0\text{ m/s}
$$

---

#### Case C - Opposing Current

$$
v_{c,x}
=
-0.5\text{ m/s}
$$

The current moves opposite the thrust direction.

The expected world-frame terminal velocity is

$$
v_{x,\infty}
=
-0.5+2.0
=
1.5\text{ m/s}
$$

while again

$$
v_{rel,\infty}
=
2.0\text{ m/s}
$$

---

### Important Physical Interpretation

The current does **not** alter the thrust capability of the vehicle.

Nor does it alter the water-relative equilibrium required to balance quadratic drag.

Instead:

$$
\boxed{
\text{current shifts ground-relative motion while preserving the thrust-drag equilibrium relative to water}
}
$$

Thus the three expected steady states are:

| Case | $v_{c,x}$ | Expected $v_{rel,\infty}$ | Expected $v_{x,\infty}$ |
|---|---:|---:|---:|
| No current | $0.0$ m/s | $2.0$ m/s | $2.0$ m/s |
| Following | $+0.5$ m/s | $2.0$ m/s | $2.5$ m/s |
| Opposing | $-0.5$ m/s | $2.0$ m/s | $1.5$ m/s |

This comparison also provides a direct **zero-current recovery test** against the canonical Notebook 05 terminal-velocity result.

---

### Numerical Integration

Each case uses the already validated HydroLab-3D explicit-Euler ordering.

At every timestep:

$$
\mathbf{v}_{rel,k}
=
\mathbf{v}_k
-
\mathbf{v}_c
$$

$$
\mathbf{F}_{D,k}
=
-C_D
\|\mathbf{v}_{rel,k}\|
\mathbf{v}_{rel,k}
$$

$$
\mathbf{F}_{net,k}
=
\mathbf{F}_{T,k}
+
\mathbf{F}_{D,k}
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

$$
\mathbf{a}_k
=
\frac{\mathbf{F}_{net,k}}{m}
$$

followed by

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

and

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

---

### Validation Criteria

This step passes only if:

1. the production thruster model produces exactly

$$
\mathbf{F}_T
=
\begin{bmatrix}
20\\
0\\
0
\end{bmatrix}
\text{ N}
$$

for the canonical command;

2. neutral buoyancy remains exact;

3. the no-current case converges to approximately

$$
v_x
=
2.0\text{ m/s}
$$

4. the following-current case converges to approximately

$$
v_x
=
2.5\text{ m/s}
$$

5. the opposing-current case converges to approximately

$$
v_x
=
1.5\text{ m/s}
$$

6. all three cases converge to the same water-relative terminal speed:

$$
v_{rel,x}
\rightarrow
2.0\text{ m/s}
$$

7. terminal drag converges to

$$
\mathbf{F}_D
\rightarrow
\begin{bmatrix}
-20\\
0\\
0
\end{bmatrix}
\text{ N}
$$

8. terminal net force approaches zero;

9. no unintended $y$ or $z$ motion develops;

10. all state and force histories remain finite.

If these conditions pass, HydroLab-3D will have demonstrated analytically and numerically that a uniform current shifts **world-frame terminal velocity** without changing the underlying **water-relative thrust-drag equilibrium**.

In [5]:
from hydrolab3d.dynamics import (
    acceleration_from_force,
    integrate_velocity,
    integrate_position,
    gravity_force,
    buoyancy_force,
    thruster_force_world,
)


# =====================================================================
# CANONICAL PARAMETERS
# =====================================================================

mass = 20.0
fluid_density = 1000.0
gravity = 9.81
displaced_volume = 0.020

drag_coefficient = 5.0

max_thrust = 40.0
command = 0.5

thruster_direction_body = np.array(
    [1.0, 0.0, 0.0],
    dtype=float,
)

roll = 0.0
pitch = 0.0
yaw = 0.0

dt = 0.01
duration = 20.0
num_steps = int(round(duration / dt))


# =====================================================================
# VALIDATED PRODUCTION THRUST
# =====================================================================

thrust_force = thruster_force_world(
    command,
    max_thrust,
    thruster_direction_body,
    roll,
    pitch,
    yaw,
)

expected_thrust = np.array(
    [20.0, 0.0, 0.0],
    dtype=float,
)

np.testing.assert_allclose(
    thrust_force,
    expected_thrust,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# HYDROSTATIC EQUILIBRIUM
# =====================================================================

gravity_vector = gravity_force(
    mass,
    gravity,
)

buoyancy_vector = buoyancy_force(
    fluid_density,
    displaced_volume,
    gravity,
)

hydrostatic_force = (
    gravity_vector
    + buoyancy_vector
)

np.testing.assert_allclose(
    hydrostatic_force,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# ANALYTICAL TERMINAL SPEED
# =====================================================================

thrust_magnitude = np.linalg.norm(
    thrust_force
)

analytical_relative_terminal_speed = np.sqrt(
    thrust_magnitude
    / drag_coefficient
)

np.testing.assert_allclose(
    analytical_relative_terminal_speed,
    2.0,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# SIMULATION HELPER
# =====================================================================

def simulate_current_case(
    case_name,
    current_velocity,
):
    position = np.array(
        [0.0, 0.0, -10.0],
        dtype=float,
    )

    velocity = np.zeros(
        3,
        dtype=float,
    )

    time_hist = [0.0]
    position_hist = [position.copy()]
    velocity_hist = [velocity.copy()]
    relative_hist = [
        relative_water_velocity(
            velocity,
            current_velocity,
        )
    ]
    drag_hist = []
    net_force_hist = []
    acceleration_hist = []

    initial_drag, _ = current_aware_drag_force(
        vehicle_velocity=velocity,
        current_velocity=current_velocity,
        drag_coefficient=drag_coefficient,
    )

    initial_net_force = (
        thrust_force
        + initial_drag
        + hydrostatic_force
    )

    initial_acceleration = acceleration_from_force(
        initial_net_force,
        mass,
    )

    drag_hist.append(
        initial_drag.copy()
    )

    net_force_hist.append(
        initial_net_force.copy()
    )

    acceleration_hist.append(
        initial_acceleration.copy()
    )

    for step in range(num_steps):

        drag_force, relative_velocity = (
            current_aware_drag_force(
                vehicle_velocity=velocity,
                current_velocity=current_velocity,
                drag_coefficient=drag_coefficient,
            )
        )

        net_force = (
            thrust_force
            + drag_force
            + hydrostatic_force
        )

        acceleration = acceleration_from_force(
            net_force,
            mass,
        )

        position = integrate_position(
            position,
            velocity,
            dt,
        )

        velocity = integrate_velocity(
            velocity,
            acceleration,
            dt,
        )

        time_hist.append(
            (step + 1) * dt
        )

        position_hist.append(
            position.copy()
        )

        velocity_hist.append(
            velocity.copy()
        )

        relative_hist.append(
            relative_water_velocity(
                velocity,
                current_velocity,
            )
        )

        drag_hist.append(
            drag_force.copy()
        )

        net_force_hist.append(
            net_force.copy()
        )

        acceleration_hist.append(
            acceleration.copy()
        )

    return {
        "name": case_name,
        "current": np.asarray(
            current_velocity,
            dtype=float,
        ),
        "time": np.asarray(time_hist),
        "position": np.asarray(position_hist),
        "velocity": np.asarray(velocity_hist),
        "relative_velocity": np.asarray(relative_hist),
        "drag": np.asarray(drag_hist),
        "net_force": np.asarray(net_force_hist),
        "acceleration": np.asarray(acceleration_hist),
    }


# =====================================================================
# RUN THREE CANONICAL CURRENT CASES
# =====================================================================

cases = {
    "zero": simulate_current_case(
        "No current",
        np.array([0.0, 0.0, 0.0]),
    ),

    "following": simulate_current_case(
        "Following current",
        np.array([0.5, 0.0, 0.0]),
    ),

    "opposing": simulate_current_case(
        "Opposing current",
        np.array([-0.5, 0.0, 0.0]),
    ),
}


# =====================================================================
# ANALYTICAL WORLD-FRAME TERMINAL VELOCITIES
# =====================================================================

expected_world_terminal = {
    "zero": 2.0,
    "following": 2.5,
    "opposing": 1.5,
}


# =====================================================================
# TERMINAL-STATE VALIDATION
# =====================================================================

terminal_tolerance = 1e-6

for key, result in cases.items():

    final_velocity = result["velocity"][-1]

    final_relative_velocity = (
        result["relative_velocity"][-1]
    )

    final_drag = result["drag"][-1]

    final_net_force = result["net_force"][-1]

    expected_world_vx = (
        expected_world_terminal[key]
    )

    # -------------------------------------------------------------
    # World-frame terminal velocity
    # -------------------------------------------------------------

    np.testing.assert_allclose(
        final_velocity[0],
        expected_world_vx,
        rtol=0.0,
        atol=terminal_tolerance,
    )

    # -------------------------------------------------------------
    # Water-relative terminal velocity
    # -------------------------------------------------------------

    np.testing.assert_allclose(
        final_relative_velocity[0],
        analytical_relative_terminal_speed,
        rtol=0.0,
        atol=terminal_tolerance,
    )

    # -------------------------------------------------------------
    # Terminal drag balances thrust
    # -------------------------------------------------------------

    np.testing.assert_allclose(
        final_drag,
        -thrust_force,
        rtol=0.0,
        atol=1e-5,
    )

    # -------------------------------------------------------------
    # Terminal net force approaches zero
    # -------------------------------------------------------------

    np.testing.assert_allclose(
        final_net_force,
        np.zeros(3),
        rtol=0.0,
        atol=1e-5,
    )

    # -------------------------------------------------------------
    # No cross-axis leakage
    # -------------------------------------------------------------

    np.testing.assert_allclose(
        result["velocity"][:, 1:],
        0.0,
        rtol=0.0,
        atol=1e-12,
    )

    np.testing.assert_allclose(
        result["position"][:, 1],
        0.0,
        rtol=0.0,
        atol=1e-12,
    )

    np.testing.assert_allclose(
        result["position"][:, 2],
        -10.0,
        rtol=0.0,
        atol=1e-12,
    )

    # -------------------------------------------------------------
    # Finite histories
    # -------------------------------------------------------------

    arrays_to_check = [
        result["time"],
        result["position"],
        result["velocity"],
        result["relative_velocity"],
        result["drag"],
        result["net_force"],
        result["acceleration"],
    ]

    assert all(
        np.all(np.isfinite(array))
        for array in arrays_to_check
    )


# =====================================================================
# COMPARATIVE VALIDATION
# =====================================================================

zero_final_vx = (
    cases["zero"]["velocity"][-1, 0]
)

following_final_vx = (
    cases["following"]["velocity"][-1, 0]
)

opposing_final_vx = (
    cases["opposing"]["velocity"][-1, 0]
)


assert (
    following_final_vx
    > zero_final_vx
    > opposing_final_vx
)


# Current should shift terminal world speed by exactly its value.
np.testing.assert_allclose(
    following_final_vx - zero_final_vx,
    0.5,
    rtol=0.0,
    atol=terminal_tolerance,
)

np.testing.assert_allclose(
    opposing_final_vx - zero_final_vx,
    -0.5,
    rtol=0.0,
    atol=terminal_tolerance,
)


# =====================================================================
# SUMMARY
# =====================================================================

print("=" * 80)
print("STEP 4 - THRUST UNDER UNIFORM-CURRENT VALIDATION")
print("=" * 80)


print("\nCanonical propulsion")
print(
    f"  Command                         : "
    f"{command:.6f}"
)
print(
    f"  Maximum thrust                  : "
    f"{max_thrust:.6f} N"
)
print(
    f"  Production thrust vector        : "
    f"{thrust_force} N"
)


print("\nAnalytical equilibrium")
print(
    f"  Expected water-relative "
    f"terminal speed                   : "
    f"{analytical_relative_terminal_speed:.12f} m/s"
)


for key in [
    "zero",
    "following",
    "opposing",
]:
    result = cases[key]

    final_velocity = (
        result["velocity"][-1]
    )

    final_relative = (
        result["relative_velocity"][-1]
    )

    final_drag = (
        result["drag"][-1]
    )

    final_net_force = (
        result["net_force"][-1]
    )

    print(
        f"\n{result['name']}"
    )

    print(
        f"  Current velocity                : "
        f"{result['current']} m/s"
    )

    print(
        f"  Expected world terminal vx      : "
        f"{expected_world_terminal[key]:.12f} m/s"
    )

    print(
        f"  Numerical world terminal vx     : "
        f"{final_velocity[0]:.12f} m/s"
    )

    print(
        f"  Numerical relative terminal vx  : "
        f"{final_relative[0]:.12f} m/s"
    )

    print(
        f"  Final drag                      : "
        f"{final_drag} N"
    )

    print(
        f"  Final net force                 : "
        f"{final_net_force} N"
    )


print("\nWorld-frame terminal-speed shifts")
print(
    f"  Following - zero current        : "
    f"{following_final_vx - zero_final_vx:.12f} m/s"
)
print(
    f"  Opposing - zero current         : "
    f"{opposing_final_vx - zero_final_vx:.12f} m/s"
)


print(
    "\n[PASS] Uniform current shifts world-frame terminal velocity "
    "while preserving the water-relative thrust-drag equilibrium."
)

STEP 4 - THRUST UNDER UNIFORM-CURRENT VALIDATION

Canonical propulsion
  Command                         : 0.500000
  Maximum thrust                  : 40.000000 N
  Production thrust vector        : [20.  0.  0.] N

Analytical equilibrium
  Expected water-relative terminal speed                   : 2.000000000000 m/s

No current
  Current velocity                : [0. 0. 0.] m/s
  Expected world terminal vx      : 2.000000000000 m/s
  Numerical world terminal vx     : 1.999999992493 m/s
  Numerical relative terminal vx  : 1.999999992493 m/s
  Final drag                      : [-19.99999985  -0.          -0.        ] N
  Final net force                 : [1.51663279e-07 0.00000000e+00 0.00000000e+00] N

Following current
  Current velocity                : [0.5 0.  0. ] m/s
  Expected world terminal vx      : 2.500000000000 m/s
  Numerical world terminal vx     : 2.499999987720 m/s
  Numerical relative terminal vx  : 1.999999987720 m/s
  Final drag                      : [-19.99999975 

## Step 5 - Inspect the Production Architecture Before Promoting the Current Model

The notebook-local ocean-current model has now passed the required physical validations:

- world-frame uniform current representation;
- water-relative velocity;
- current-aware quadratic drag;
- passive entrainment;
- following-current equilibrium;
- opposing-current equilibrium;
- zero-current recovery.

The reusable current logic is therefore mature enough to consider promotion into the HydroLab-3D source package.

However, HydroLab-3D follows a strict rule:

> **Never overwrite or introduce production files blindly. Inspect the existing repository structure first.**

This step therefore performs **architecture inspection only**.

No production source file will be created or modified yet.

---

### Why Inspection Is Required

The README proposes an eventual environmental layer resembling:

```text
src/hydrolab3d/
└── world/
    ├── __init__.py
    ├── ocean.py
    ├── currents.py
    ├── terrain.py
    └── objects.py
```

but this structure is explicitly tentative.

The current repository may contain:

- an existing `world/` package;
- an existing `currents.py`;
- an existing `world/__init__.py`;
- tests that already establish naming conventions;
- package exports that should be preserved;
- or no world layer at all yet.

The correct production design must follow the **actual repository**, not an architectural sketch.

---

### Candidate Production Responsibility

Based on the validated notebook behavior, the smallest reusable current API currently requires only two responsibilities:

1. represent and validate a uniform world-frame current;

2. compute vehicle velocity relative to the surrounding water.

Conceptually:

$$
\mathbf{v}_c
=
\text{uniform current}
$$

and

$$
\boxed{
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
}
$$

The existing production drag model should remain unchanged.

The intended runtime composition remains:

```text
world.current
      ↓
relative_water_velocity()
      ↓
quadratic_drag_force()
      ↓
vehicle force balance
```

This keeps environmental state separate from hydrodynamic force calculation.

---

### What This Step Will Inspect

The code cell will inspect:

- `src/hydrolab3d/`
- `src/hydrolab3d/world/`, if it exists;
- `src/hydrolab3d/world/__init__.py`, if it exists;
- `src/hydrolab3d/world/currents.py`, if it exists;
- `src/hydrolab3d/dynamics/__init__.py`;
- the current `tests/` directory;
- any existing current-related test file;
- relevant package-level exports.

The inspection will print actual file contents where those files already exist.

---

### Decision Criterion

After inspecting the output, we will determine whether the validated notebook-local functions should be promoted into:

`src/hydrolab3d/world/currents.py`

or whether the current repository structure indicates a better existing location.

A new module will only be introduced if it is consistent with the real package architecture.

Similarly, automated tests will only be added after inspecting the existing test layout and conventions.

---

### Expected Result

This step should produce a concise repository report showing the current state of the relevant production paths.

There is no physical numerical result to validate here.

Success means we know exactly:

- whether the `world` package already exists;
- whether `currents.py` already exists;
- what its contents are, if present;
- how package exports are currently structured;
- how existing tests are organized;
- and what files may safely be created or extended during production promotion.

Only after this inspection passes will Notebook 06 modify production code.

In [6]:
from pathlib import Path


# =====================================================================
# REPOSITORY PATHS
# =====================================================================

repo_root = Path(repo_root).resolve()

src_root = repo_root / "src"
package_root = src_root / "hydrolab3d"

world_dir = package_root / "world"
world_init = world_dir / "__init__.py"
currents_file = world_dir / "currents.py"

dynamics_init = (
    package_root
    / "dynamics"
    / "__init__.py"
)

tests_dir = repo_root / "tests"
currents_test = tests_dir / "test_currents.py"


# =====================================================================
# HELPERS
# =====================================================================

def print_section(title):
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)


def print_file_if_exists(path):
    """
    Print a repository-relative file and its contents without
    modifying anything.
    """
    relative_path = path.relative_to(repo_root)

    print(f"\n--- {relative_path} ---")

    if not path.exists():
        print("[DOES NOT EXIST]")
        return

    if not path.is_file():
        print("[EXISTS BUT IS NOT A FILE]")
        return

    content = path.read_text(
        encoding="utf-8"
    )

    if content.strip():
        print(content)
    else:
        print("[EMPTY FILE]")


# =====================================================================
# PACKAGE STRUCTURE
# =====================================================================

print_section(
    "STEP 5 - HYDROLAB-3D PRODUCTION ARCHITECTURE INSPECTION"
)

print(f"Repository root : {repo_root}")
print(f"Package root    : {package_root}")


assert package_root.is_dir(), (
    f"HydroLab-3D package directory does not exist: {package_root}"
)


print_section(
    "HYDROLAB-3D TOP-LEVEL PACKAGE CONTENTS"
)

for path in sorted(package_root.iterdir()):
    marker = "DIR " if path.is_dir() else "FILE"

    print(
        f"[{marker}] "
        f"{path.relative_to(repo_root)}"
    )


# =====================================================================
# WORLD PACKAGE INSPECTION
# =====================================================================

print_section(
    "WORLD PACKAGE"
)

if world_dir.exists():

    print(
        f"[EXISTS] "
        f"{world_dir.relative_to(repo_root)}"
    )

    if world_dir.is_dir():

        contents = sorted(
            world_dir.iterdir()
        )

        if contents:
            for path in contents:
                marker = (
                    "DIR "
                    if path.is_dir()
                    else "FILE"
                )

                print(
                    f"[{marker}] "
                    f"{path.relative_to(repo_root)}"
                )
        else:
            print(
                "[EMPTY DIRECTORY]"
            )

else:

    print(
        f"[DOES NOT EXIST] "
        f"{world_dir.relative_to(repo_root)}"
    )


print_file_if_exists(
    world_init
)

print_file_if_exists(
    currents_file
)


# =====================================================================
# EXISTING DYNAMICS EXPORTS
# =====================================================================

print_section(
    "CURRENT DYNAMICS PACKAGE EXPORTS"
)

print_file_if_exists(
    dynamics_init
)


# =====================================================================
# TEST SUITE STRUCTURE
# =====================================================================

print_section(
    "TEST DIRECTORY"
)

if not tests_dir.is_dir():
    raise FileNotFoundError(
        f"Expected tests directory not found: {tests_dir}"
    )


test_files = sorted(
    tests_dir.glob("test_*.py")
)

for path in test_files:
    print(
        f"[TEST] "
        f"{path.relative_to(repo_root)}"
    )


print_file_if_exists(
    currents_test
)


# =====================================================================
# CURRENT-RELATED SOURCE SEARCH
# =====================================================================

print_section(
    "EXISTING CURRENT-RELATED SOURCE REFERENCES"
)

search_terms = (
    "current",
    "relative_water_velocity",
    "uniform_current",
)


matches = []

for path in package_root.rglob("*.py"):

    try:
        text = path.read_text(
            encoding="utf-8"
        )
    except UnicodeDecodeError:
        continue

    lower_text = text.lower()

    matched_terms = [
        term
        for term in search_terms
        if term.lower() in lower_text
    ]

    if matched_terms:
        matches.append(
            (
                path.relative_to(repo_root),
                matched_terms,
            )
        )


if matches:

    for relative_path, matched_terms in matches:
        print(
            f"{relative_path}"
            f"  ->  "
            f"{', '.join(matched_terms)}"
        )

else:

    print(
        "[NONE] No current-related implementation "
        "found in the production package."
    )


# =====================================================================
# SUMMARY
# =====================================================================

print_section(
    "INSPECTION SUMMARY"
)

print(
    f"world/ exists              : "
    f"{world_dir.is_dir()}"
)

print(
    f"world/__init__.py exists   : "
    f"{world_init.is_file()}"
)

print(
    f"world/currents.py exists   : "
    f"{currents_file.is_file()}"
)

print(
    f"tests/test_currents.py     : "
    f"{currents_test.is_file()}"
)

print(
    f"Current-related source hits: "
    f"{len(matches)}"
)


print(
    "\n[PASS] Production targets inspected. "
    "No repository files were modified."
)


STEP 5 - HYDROLAB-3D PRODUCTION ARCHITECTURE INSPECTION
Repository root : /home/agniacolyte/HydroLab-3D
Package root    : /home/agniacolyte/HydroLab-3D/src/hydrolab3d

HYDROLAB-3D TOP-LEVEL PACKAGE CONTENTS
[FILE] src/hydrolab3d/__init__.py
[DIR ] src/hydrolab3d/__pycache__
[DIR ] src/hydrolab3d/communication
[DIR ] src/hydrolab3d/control
[DIR ] src/hydrolab3d/core
[DIR ] src/hydrolab3d/dynamics
[DIR ] src/hydrolab3d/interface
[DIR ] src/hydrolab3d/metrics
[DIR ] src/hydrolab3d/sensors
[DIR ] src/hydrolab3d/utils
[DIR ] src/hydrolab3d/vehicles
[DIR ] src/hydrolab3d/visualization
[DIR ] src/hydrolab3d/world

WORLD PACKAGE
[EXISTS] src/hydrolab3d/world
[FILE] src/hydrolab3d/world/__init__.py
[FILE] src/hydrolab3d/world/currents.py
[FILE] src/hydrolab3d/world/objects.py
[FILE] src/hydrolab3d/world/ocean.py
[FILE] src/hydrolab3d/world/terrain.py

--- src/hydrolab3d/world/__init__.py ---
[EMPTY FILE]

--- src/hydrolab3d/world/currents.py ---
[EMPTY FILE]

CURRENT DYNAMICS PACKAGE EXPORTS



## Step 6 - Promote the Validated Ocean-Current Primitives to Production

The notebook-local current implementation has now passed its required prototype validations, and Step 5 confirmed that HydroLab-3D already contains the intended production locations:

`src/hydrolab3d/world/currents.py`

`src/hydrolab3d/world/__init__.py`

and

`tests/test_currents.py`

All three files currently exist but are empty.

We can therefore promote the validated current primitives without introducing a new package structure or modifying the existing dynamics layer.

---

### Production Responsibility

The world-current module will own two reusable operations.

#### Uniform Current Representation

The environmental model will validate and return a constant world-frame current vector:

$$
\mathbf{v}_c
=
\begin{bmatrix}
v_{c,x}\\
v_{c,y}\\
v_{c,z}
\end{bmatrix}
$$

The function will require an exact finite three-vector and will return an independent array rather than aliasing caller-owned storage.

---

#### Water-Relative Velocity

The module will also provide:

$$
\boxed{
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
}
$$

where both quantities are expressed in the HydroLab-3D world frame.

This operation belongs naturally beside the current model because it expresses the relationship between vehicle motion and environmental flow.

---

### Deliberate Non-Responsibility

The production current module will **not** calculate hydrodynamic drag.

The existing validated drag implementation remains:

`hydrolab3d.dynamics.quadratic_drag_force`

The intended production composition is therefore:

    world current
         ↓
    relative_water_velocity()
         ↓
    quadratic_drag_force()
         ↓
    vehicle force balance

This preserves separation between:

- environmental state;
- vehicle-water relative motion;
- hydrodynamic force calculation.

No validated Notebook 03 drag code needs modification.

---

### Public World API

The two current functions will be exported through:

`hydrolab3d.world`

so future HydroLab-3D code can use:

`uniform_current_velocity`

and:

`relative_water_velocity`

without depending on the internal module path.

---

### Automated Regression Tests

The existing empty:

`tests/test_currents.py`

will receive focused tests covering the reusable current API.

The tests will verify:

1. zero current;
2. known uniform 3D current;
3. independent returned storage;
4. known water-relative velocity;
5. stationary vehicle in moving water;
6. vehicle moving exactly with the current;
7. arbitrary 3D relative velocity;
8. malformed current-vector rejection;
9. malformed vehicle-vector rejection;
10. non-finite current rejection;
11. non-finite vehicle-velocity rejection;
12. clean composition with the existing quadratic-drag implementation.

The drag-composition test is important because it verifies the intended module boundary without introducing a second drag implementation.

---

### Promotion Rule

This step will write only the three files that were explicitly inspected in Step 5:

- `src/hydrolab3d/world/currents.py`
- `src/hydrolab3d/world/__init__.py`
- `tests/test_currents.py`

No other production file will be modified.

After writing them, this step will display the complete resulting contents so the exact promoted implementation is visible in the notebook record.

The next step will then perform a **fresh production re-import**, verify module resolution, rerun the canonical physical cases through the promoted implementation, and execute the focused current regression tests.

In [7]:
from pathlib import Path


# =====================================================================
# STEP 6 - PRODUCTION PROMOTION
# =====================================================================

currents_source = '''"""Ocean-current utilities for HydroLab-3D."""

import numpy as np


def uniform_current_velocity(current_velocity):
    """
    Return a validated uniform ocean-current velocity.

    The current velocity is expressed in the HydroLab-3D world frame.

    Parameters
    ----------
    current_velocity : array-like
        Uniform world-frame current velocity [vc_x, vc_y, vc_z] in m/s.

    Returns
    -------
    np.ndarray
        Independent finite 3-vector containing the current velocity.

    Raises
    ------
    ValueError
        If current_velocity does not have shape (3,) or contains
        non-finite values.
    """
    current_velocity = np.asarray(
        current_velocity,
        dtype=float,
    )

    if current_velocity.shape != (3,):
        raise ValueError(
            "current_velocity must have shape (3,)"
        )

    if not np.all(np.isfinite(current_velocity)):
        raise ValueError(
            "current_velocity must contain only finite values"
        )

    return current_velocity.copy()


def relative_water_velocity(
    vehicle_velocity,
    current_velocity,
):
    """
    Compute vehicle velocity relative to the surrounding water.

    Both velocity vectors are expressed in the HydroLab-3D world frame.

    Parameters
    ----------
    vehicle_velocity : array-like
        Vehicle world-frame velocity [vx, vy, vz] in m/s.

    current_velocity : array-like
        Ocean-current world-frame velocity [vc_x, vc_y, vc_z] in m/s.

    Returns
    -------
    np.ndarray
        Water-relative velocity:

            v_rel = v_vehicle - v_current

    Raises
    ------
    ValueError
        If either input does not have shape (3,) or contains
        non-finite values.
    """
    vehicle_velocity = np.asarray(
        vehicle_velocity,
        dtype=float,
    )

    current_velocity = np.asarray(
        current_velocity,
        dtype=float,
    )

    if vehicle_velocity.shape != (3,):
        raise ValueError(
            "vehicle_velocity must have shape (3,)"
        )

    if current_velocity.shape != (3,):
        raise ValueError(
            "current_velocity must have shape (3,)"
        )

    if not np.all(np.isfinite(vehicle_velocity)):
        raise ValueError(
            "vehicle_velocity must contain only finite values"
        )

    if not np.all(np.isfinite(current_velocity)):
        raise ValueError(
            "current_velocity must contain only finite values"
        )

    return vehicle_velocity - current_velocity
'''


world_init_source = '''"""World and environmental utilities for HydroLab-3D."""

from .currents import (
    relative_water_velocity,
    uniform_current_velocity,
)

__all__ = [
    "relative_water_velocity",
    "uniform_current_velocity",
]
'''


currents_test_source = '''import numpy as np
import pytest

from hydrolab3d.dynamics import quadratic_drag_force
from hydrolab3d.world import (
    relative_water_velocity,
    uniform_current_velocity,
)


def test_zero_uniform_current():
    current = uniform_current_velocity(
        np.zeros(3)
    )

    np.testing.assert_array_equal(
        current,
        np.zeros(3),
    )


def test_known_uniform_current():
    expected = np.array(
        [0.5, 0.25, -0.1],
        dtype=float,
    )

    current = uniform_current_velocity(
        expected
    )

    np.testing.assert_array_equal(
        current,
        expected,
    )


def test_uniform_current_returns_independent_array():
    source = np.array(
        [0.5, 0.25, -0.1],
        dtype=float,
    )

    current = uniform_current_velocity(
        source
    )

    assert not np.shares_memory(
        current,
        source,
    )


def test_known_relative_water_velocity():
    vehicle = np.array(
        [2.0, -1.0, 0.5],
        dtype=float,
    )

    current = np.array(
        [0.5, 0.25, -0.1],
        dtype=float,
    )

    expected = np.array(
        [1.5, -1.25, 0.6],
        dtype=float,
    )

    actual = relative_water_velocity(
        vehicle,
        current,
    )

    np.testing.assert_allclose(
        actual,
        expected,
        rtol=0.0,
        atol=1e-12,
    )


def test_stationary_vehicle_relative_velocity():
    current = np.array(
        [0.5, -0.25, 0.1],
        dtype=float,
    )

    relative = relative_water_velocity(
        np.zeros(3),
        current,
    )

    np.testing.assert_allclose(
        relative,
        -current,
        rtol=0.0,
        atol=1e-12,
    )


def test_comoving_vehicle_has_zero_relative_velocity():
    current = np.array(
        [0.5, -0.25, 0.1],
        dtype=float,
    )

    relative = relative_water_velocity(
        current,
        current,
    )

    np.testing.assert_array_equal(
        relative,
        np.zeros(3),
    )


def test_arbitrary_3d_relative_velocity():
    vehicle = np.array(
        [-1.2, 3.4, -0.8],
        dtype=float,
    )

    current = np.array(
        [0.3, -0.6, 0.2],
        dtype=float,
    )

    relative = relative_water_velocity(
        vehicle,
        current,
    )

    np.testing.assert_allclose(
        relative,
        vehicle - current,
        rtol=0.0,
        atol=1e-12,
    )


@pytest.mark.parametrize(
    "invalid_current",
    [
        np.ones(2),
        np.ones(4),
        np.ones((3, 1)),
    ],
)
def test_uniform_current_rejects_invalid_shape(
    invalid_current,
):
    with pytest.raises(
        ValueError,
        match=r"shape \\(3,\\)",
    ):
        uniform_current_velocity(
            invalid_current
        )


@pytest.mark.parametrize(
    "invalid_current",
    [
        np.array([0.0, np.nan, 0.0]),
        np.array([0.0, np.inf, 0.0]),
        np.array([0.0, -np.inf, 0.0]),
    ],
)
def test_uniform_current_rejects_nonfinite_values(
    invalid_current,
):
    with pytest.raises(
        ValueError,
        match="finite",
    ):
        uniform_current_velocity(
            invalid_current
        )


@pytest.mark.parametrize(
    "invalid_vehicle",
    [
        np.ones(2),
        np.ones(4),
        np.ones((3, 1)),
    ],
)
def test_relative_velocity_rejects_invalid_vehicle_shape(
    invalid_vehicle,
):
    with pytest.raises(
        ValueError,
        match=r"vehicle_velocity must have shape \\(3,\\)",
    ):
        relative_water_velocity(
            invalid_vehicle,
            np.zeros(3),
        )


@pytest.mark.parametrize(
    "invalid_current",
    [
        np.ones(2),
        np.ones(4),
        np.ones((3, 1)),
    ],
)
def test_relative_velocity_rejects_invalid_current_shape(
    invalid_current,
):
    with pytest.raises(
        ValueError,
        match=r"current_velocity must have shape \\(3,\\)",
    ):
        relative_water_velocity(
            np.zeros(3),
            invalid_current,
        )


@pytest.mark.parametrize(
    "invalid_vehicle",
    [
        np.array([0.0, np.nan, 0.0]),
        np.array([0.0, np.inf, 0.0]),
        np.array([0.0, -np.inf, 0.0]),
    ],
)
def test_relative_velocity_rejects_nonfinite_vehicle(
    invalid_vehicle,
):
    with pytest.raises(
        ValueError,
        match="finite",
    ):
        relative_water_velocity(
            invalid_vehicle,
            np.zeros(3),
        )


@pytest.mark.parametrize(
    "invalid_current",
    [
        np.array([0.0, np.nan, 0.0]),
        np.array([0.0, np.inf, 0.0]),
        np.array([0.0, -np.inf, 0.0]),
    ],
)
def test_relative_velocity_rejects_nonfinite_current(
    invalid_current,
):
    with pytest.raises(
        ValueError,
        match="finite",
    ):
        relative_water_velocity(
            np.zeros(3),
            invalid_current,
        )


def test_current_relative_velocity_composes_with_drag():
    vehicle = np.zeros(3)

    current = np.array(
        [0.5, 0.25, -0.1],
        dtype=float,
    )

    drag_coefficient = 3.0

    relative = relative_water_velocity(
        vehicle,
        current,
    )

    drag = quadratic_drag_force(
        relative,
        drag_coefficient,
    )

    expected_drag = (
        -drag_coefficient
        * np.linalg.norm(relative)
        * relative
    )

    np.testing.assert_allclose(
        drag,
        expected_drag,
        rtol=0.0,
        atol=1e-12,
    )

    assert np.dot(
        drag,
        current,
    ) > 0.0
'''


# =====================================================================
# WRITE ONLY THE THREE INSPECTED FILES
# =====================================================================

currents_file.write_text(
    currents_source,
    encoding="utf-8",
)

world_init.write_text(
    world_init_source,
    encoding="utf-8",
)

currents_test.write_text(
    currents_test_source,
    encoding="utf-8",
)


# =====================================================================
# VERIFY FILE CREATION
# =====================================================================

assert currents_file.is_file()
assert world_init.is_file()
assert currents_test.is_file()

assert currents_file.read_text(
    encoding="utf-8"
) == currents_source

assert world_init.read_text(
    encoding="utf-8"
) == world_init_source

assert currents_test.read_text(
    encoding="utf-8"
) == currents_test_source


# =====================================================================
# DISPLAY PROMOTED CONTENT
# =====================================================================

print("=" * 80)
print("STEP 6 - OCEAN-CURRENT PRODUCTION PROMOTION")
print("=" * 80)


for path in [
    currents_file,
    world_init,
    currents_test,
]:

    print()
    print(
        "-" * 80
    )

    print(
        path.relative_to(repo_root)
    )

    print(
        "-" * 80
    )

    print(
        path.read_text(
            encoding="utf-8"
        )
    )


print("=" * 80)
print("PROMOTION SUMMARY")
print("=" * 80)

print(
    "Created/updated production module : "
    "src/hydrolab3d/world/currents.py"
)

print(
    "Created/updated world exports     : "
    "src/hydrolab3d/world/__init__.py"
)

print(
    "Created/updated focused tests     : "
    "tests/test_currents.py"
)

print(
    "\n[PASS] Validated notebook current primitives "
    "have been promoted to the HydroLab-3D production package."
)

print(
    "[NEXT] Freshly re-import production code and "
    "rerun canonical physical validation before regression testing."
)

STEP 6 - OCEAN-CURRENT PRODUCTION PROMOTION

--------------------------------------------------------------------------------
src/hydrolab3d/world/currents.py
--------------------------------------------------------------------------------
"""Ocean-current utilities for HydroLab-3D."""

import numpy as np


def uniform_current_velocity(current_velocity):
    """
    Return a validated uniform ocean-current velocity.

    The current velocity is expressed in the HydroLab-3D world frame.

    Parameters
    ----------
    current_velocity : array-like
        Uniform world-frame current velocity [vc_x, vc_y, vc_z] in m/s.

    Returns
    -------
    np.ndarray
        Independent finite 3-vector containing the current velocity.

    Raises
    ------
    ValueError
        If current_velocity does not have shape (3,) or contains
        non-finite values.
    """
    current_velocity = np.asarray(
        current_velocity,
        dtype=float,
    )

    if current_velocity.shape != (3,

## Step 7 - Re-Import the Production Current Model and Validate the Promoted Implementation

The ocean-current primitives have now been promoted into:

`src/hydrolab3d/world/currents.py`

and exported through:

`hydrolab3d.world`

The next requirement is critical:

> The notebook must stop relying on the notebook-local prototype and prove that the **production implementation itself** reproduces the validated behavior.

This step therefore deliberately re-imports the promoted current functions and reruns the canonical physical validation through production code.

It also executes the focused automated regression suite in:

`tests/test_currents.py`

No production source files are modified in this step.

---

### Why Re-Import Is Necessary

A successful prototype does not prove that the promoted package implementation is correct.

Possible promotion failures include:

- transcription mistakes;
- wrong package exports;
- stale notebook imports;
- importing an unintended module;
- test-discovery failures;
- source-path mistakes.

Therefore the promoted implementation must be treated as a new dependency and independently exercised.

---

### Production Functions Under Validation

The production world package should now expose:

`uniform_current_velocity()`

and

`relative_water_velocity()`

from:

`src/hydrolab3d/world/currents.py`

The existing production drag implementation remains:

`quadratic_drag_force()`

from:

`src/hydrolab3d/dynamics/drag.py`

The intended production composition is:

```text
uniform current
      ↓
relative_water_velocity()
      ↓
quadratic_drag_force()
      ↓
reduced vehicle dynamics
```

---

### Canonical Production Validation

The promoted implementation will be exercised using the same core physical cases already validated during prototyping.

#### 1. Uniform Current

The production current function must reproduce:

$$
\mathbf{v}_c
=
\begin{bmatrix}
0.5\\
-0.25\\
0.1
\end{bmatrix}
\mathrm{m/s}
$$

exactly.

---

#### 2. Passive Drift

A neutrally buoyant passive vehicle beginning from

$$
\mathbf{v}(0)
=
\mathbf{0}
$$

inside the uniform current must satisfy:

$$
\mathbf{v}
\rightarrow
\mathbf{v}_c
$$

while:

$$
\mathbf{v}_{rel}
=
\mathbf{v}-\mathbf{v}_c
\rightarrow
\mathbf{0}
$$

The numerical relative-speed history will again be compared with the analytical quadratic-drag solution:

$$
s(t)
=
\frac{s_0}{
1+
\frac{C_D}{m}s_0t
}
$$

---

#### 3. Thrust Under Current

Using the canonical parameters

$$
F_T
=
20\text{ N}
$$

and

$$
C_D
=
5
$$

the water-relative terminal speed must remain:

$$
v_{rel,\infty}
=
2.0\text{ m/s}
$$

The expected world-frame terminal velocities remain:

$$
v_{x,\infty}
=
2.0\text{ m/s}
$$

for zero current,

$$
v_{x,\infty}
=
2.5\text{ m/s}
$$

for a following current of

$$
v_{c,x}
=
0.5\text{ m/s}
$$

and

$$
v_{x,\infty}
=
1.5\text{ m/s}
$$

for an opposing current of

$$
v_{c,x}
=
-0.5\text{ m/s}
$$

---

### Fresh-Process Import Check

In addition to the notebook's explicit module reload, this step will launch a fresh Python interpreter and import:

`hydrolab3d.world`

from the repository source tree.

This verifies that the package does not work merely because of stale state already loaded into the notebook kernel.

---

### Focused Regression

The current-specific automated tests will then be executed using:

`pytest tests/test_currents.py`

This gives a direct regression result for the newly promoted environmental-current layer.

The complete HydroLab-3D regression suite will remain a separate final implementation step so that any failure can be isolated cleanly.

---

### Validation Criteria

This step passes only if:

1. the re-imported current functions resolve to the intended production module;

2. the fresh Python process imports the public `hydrolab3d.world` API successfully;

3. the production uniform-current function reproduces the configured vector;

4. production passive drift reproduces the expected entrainment behavior;

5. the production passive-drift relative-speed history agrees closely with the analytical solution;

6. zero, following, and opposing current cases reproduce their expected terminal world velocities;

7. all three thrust cases retain approximately

$$
2.0\text{ m/s}
$$

water-relative terminal velocity;

8. the focused `tests/test_currents.py` regression suite passes completely.

If these checks succeed, the promoted current implementation can be treated as a validated production dependency rather than a notebook-only prototype.

In [8]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

import numpy as np


# =====================================================================
# STEP 7 - FORCE PRODUCTION RE-IMPORT
# =====================================================================

import hydrolab3d.world.currents as production_currents

production_currents = importlib.reload(
    production_currents
)

from hydrolab3d.world.currents import (
    relative_water_velocity as production_relative_water_velocity,
    uniform_current_velocity as production_uniform_current_velocity,
)


production_module_path = Path(
    production_currents.__file__
).resolve()

expected_production_module_path = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "world"
    / "currents.py"
).resolve()


assert (
    production_module_path
    == expected_production_module_path
), (
    "Production current module resolved to the wrong file.\n"
    f"Expected: {expected_production_module_path}\n"
    f"Actual:   {production_module_path}"
)


print("=" * 80)
print("STEP 7 - PRODUCTION CURRENT RE-IMPORT AND VALIDATION")
print("=" * 80)

print(
    "\nProduction current module:"
)
print(
    f"  {production_module_path.relative_to(repo_root)}"
)

print(
    "[PASS] Current functions explicitly re-imported "
    "from the promoted production module."
)


# =====================================================================
# PRODUCTION CURRENT PRIMITIVE CHECK
# =====================================================================

canonical_current = (
    production_uniform_current_velocity(
        [0.5, -0.25, 0.1]
    )
)

expected_canonical_current = np.array(
    [0.5, -0.25, 0.1],
    dtype=float,
)

np.testing.assert_array_equal(
    canonical_current,
    expected_canonical_current,
)


# =====================================================================
# PRODUCTION CURRENT-AWARE DRAG COMPOSITION
# =====================================================================

def production_current_aware_drag_force(
    vehicle_velocity,
    current_velocity,
    drag_coefficient,
):
    relative_velocity = (
        production_relative_water_velocity(
            vehicle_velocity,
            current_velocity,
        )
    )

    drag_force = quadratic_drag_force(
        relative_velocity,
        drag_coefficient,
    )

    return drag_force, relative_velocity


# =====================================================================
# CANONICAL PASSIVE-DRIFT VALIDATION THROUGH PRODUCTION CODE
# =====================================================================

mass = 20.0
fluid_density = 1000.0
gravity = 9.81
displaced_volume = 0.020
drag_coefficient = 5.0

dt = 0.01
duration = 120.0
num_steps = int(round(duration / dt))


gravity_vector = gravity_force(
    mass,
    gravity,
)

buoyancy_vector = buoyancy_force(
    fluid_density,
    displaced_volume,
    gravity,
)

hydrostatic_force = (
    gravity_vector
    + buoyancy_vector
)

np.testing.assert_allclose(
    hydrostatic_force,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


position = np.array(
    [0.0, 0.0, -10.0],
    dtype=float,
)

velocity = np.zeros(
    3,
    dtype=float,
)


passive_time = [0.0]
passive_relative_speed = [
    np.linalg.norm(
        production_relative_water_velocity(
            velocity,
            canonical_current,
        )
    )
]


initial_relative_speed = (
    passive_relative_speed[0]
)


for step in range(num_steps):

    drag_force, relative_velocity = (
        production_current_aware_drag_force(
            vehicle_velocity=velocity,
            current_velocity=canonical_current,
            drag_coefficient=drag_coefficient,
        )
    )

    net_force = (
        drag_force
        + hydrostatic_force
    )

    acceleration = acceleration_from_force(
        net_force,
        mass,
    )

    position = integrate_position(
        position,
        velocity,
        dt,
    )

    velocity = integrate_velocity(
        velocity,
        acceleration,
        dt,
    )

    passive_time.append(
        (step + 1) * dt
    )

    passive_relative_speed.append(
        np.linalg.norm(
            production_relative_water_velocity(
                velocity,
                canonical_current,
            )
        )
    )


passive_time = np.asarray(
    passive_time
)

passive_relative_speed = np.asarray(
    passive_relative_speed
)


analytical_passive_speed = (
    initial_relative_speed
    / (
        1.0
        + (
            drag_coefficient
            / mass
        )
        * initial_relative_speed
        * passive_time
    )
)


max_passive_error = np.max(
    np.abs(
        passive_relative_speed
        - analytical_passive_speed
    )
)


final_passive_velocity = velocity.copy()

final_passive_relative_velocity = (
    production_relative_water_velocity(
        final_passive_velocity,
        canonical_current,
    )
)


assert (
    np.linalg.norm(
        final_passive_relative_velocity
    )
    <
    initial_relative_speed
)

assert np.all(
    np.diff(
        passive_relative_speed
    )
    <= 1e-12
)

assert (
    max_passive_error
    < 5e-4
)


# =====================================================================
# CANONICAL THRUST/CURRENT VALIDATION THROUGH PRODUCTION CODE
# =====================================================================

max_thrust = 40.0
command = 0.5

thruster_direction_body = np.array(
    [1.0, 0.0, 0.0],
    dtype=float,
)

thrust_force = thruster_force_world(
    command,
    max_thrust,
    thruster_direction_body,
    0.0,
    0.0,
    0.0,
)

np.testing.assert_allclose(
    thrust_force,
    np.array([20.0, 0.0, 0.0]),
    rtol=0.0,
    atol=1e-12,
)


analytical_relative_terminal_speed = np.sqrt(
    np.linalg.norm(thrust_force)
    / drag_coefficient
)


def production_terminal_velocity_case(
    current_velocity,
):
    current_velocity = (
        production_uniform_current_velocity(
            current_velocity
        )
    )

    position = np.array(
        [0.0, 0.0, -10.0],
        dtype=float,
    )

    velocity = np.zeros(
        3,
        dtype=float,
    )

    thrust_duration = 20.0
    thrust_steps = int(
        round(
            thrust_duration
            / dt
        )
    )

    final_drag = None
    final_net_force = None

    for _ in range(thrust_steps):

        drag_force, relative_velocity = (
            production_current_aware_drag_force(
                vehicle_velocity=velocity,
                current_velocity=current_velocity,
                drag_coefficient=drag_coefficient,
            )
        )

        net_force = (
            thrust_force
            + drag_force
            + hydrostatic_force
        )

        acceleration = acceleration_from_force(
            net_force,
            mass,
        )

        position = integrate_position(
            position,
            velocity,
            dt,
        )

        velocity = integrate_velocity(
            velocity,
            acceleration,
            dt,
        )

        final_drag = drag_force
        final_net_force = net_force

    final_relative_velocity = (
        production_relative_water_velocity(
            velocity,
            current_velocity,
        )
    )

    return {
        "current": current_velocity,
        "velocity": velocity,
        "relative_velocity": final_relative_velocity,
        "drag": final_drag,
        "net_force": final_net_force,
        "position": position,
    }


production_cases = {
    "zero": production_terminal_velocity_case(
        [0.0, 0.0, 0.0]
    ),
    "following": production_terminal_velocity_case(
        [0.5, 0.0, 0.0]
    ),
    "opposing": production_terminal_velocity_case(
        [-0.5, 0.0, 0.0]
    ),
}


expected_terminal_vx = {
    "zero": 2.0,
    "following": 2.5,
    "opposing": 1.5,
}


for key, result in production_cases.items():

    np.testing.assert_allclose(
        result["velocity"][0],
        expected_terminal_vx[key],
        rtol=0.0,
        atol=1e-6,
    )

    np.testing.assert_allclose(
        result["relative_velocity"][0],
        analytical_relative_terminal_speed,
        rtol=0.0,
        atol=1e-6,
    )

    np.testing.assert_allclose(
        result["velocity"][1:],
        np.zeros(2),
        rtol=0.0,
        atol=1e-12,
    )

    np.testing.assert_allclose(
        result["drag"],
        -thrust_force,
        rtol=0.0,
        atol=1e-5,
    )

    np.testing.assert_allclose(
        result["net_force"],
        np.zeros(3),
        rtol=0.0,
        atol=1e-5,
    )


# =====================================================================
# FRESH-PROCESS PUBLIC API IMPORT
# =====================================================================

fresh_process_code = """
from hydrolab3d.world import (
    relative_water_velocity,
    uniform_current_velocity,
)

current = uniform_current_velocity([0.5, -0.25, 0.1])
relative = relative_water_velocity([0.0, 0.0, 0.0], current)

print("current =", current)
print("relative =", relative)
print("FRESH_IMPORT_PASS")
"""


fresh_env = os.environ.copy()

existing_pythonpath = fresh_env.get(
    "PYTHONPATH",
    "",
)

fresh_env["PYTHONPATH"] = (
    str(src_path)
    if not existing_pythonpath
    else (
        str(src_path)
        + os.pathsep
        + existing_pythonpath
    )
)


fresh_result = subprocess.run(
    [
        sys.executable,
        "-c",
        fresh_process_code,
    ],
    cwd=repo_root,
    env=fresh_env,
    capture_output=True,
    text=True,
)


if fresh_result.returncode != 0:
    print(
        fresh_result.stdout
    )

    print(
        fresh_result.stderr
    )

    raise RuntimeError(
        "Fresh-process HydroLab-3D world import failed."
    )


assert (
    "FRESH_IMPORT_PASS"
    in fresh_result.stdout
)


# =====================================================================
# FOCUSED CURRENT REGRESSION
# =====================================================================

pytest_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_currents.py",
        "-q",
    ],
    cwd=repo_root,
    env=fresh_env,
    capture_output=True,
    text=True,
)


if pytest_result.returncode != 0:

    print(
        pytest_result.stdout
    )

    print(
        pytest_result.stderr
    )

    raise RuntimeError(
        "Focused ocean-current regression failed."
    )


# =====================================================================
# SUMMARY
# =====================================================================

print()
print("=" * 80)
print("PRODUCTION PASSIVE-DRIFT VALIDATION")
print("=" * 80)

print(
    f"Current velocity                  : "
    f"{canonical_current} m/s"
)

print(
    f"Final vehicle velocity            : "
    f"{final_passive_velocity} m/s"
)

print(
    f"Final relative velocity           : "
    f"{final_passive_relative_velocity} m/s"
)

print(
    f"Numerical final relative speed    : "
    f"{passive_relative_speed[-1]:.12f} m/s"
)

print(
    f"Analytical final relative speed   : "
    f"{analytical_passive_speed[-1]:.12f} m/s"
)

print(
    f"Maximum relative-speed error      : "
    f"{max_passive_error:.12e} m/s"
)


print()
print("=" * 80)
print("PRODUCTION THRUST/CURRENT VALIDATION")
print("=" * 80)

for key in [
    "zero",
    "following",
    "opposing",
]:
    result = production_cases[key]

    print(
        f"\n{key.capitalize()} current"
    )

    print(
        f"  Current velocity               : "
        f"{result['current']} m/s"
    )

    print(
        f"  Expected terminal vx           : "
        f"{expected_terminal_vx[key]:.12f} m/s"
    )

    print(
        f"  Numerical terminal vx          : "
        f"{result['velocity'][0]:.12f} m/s"
    )

    print(
        f"  Relative terminal vx           : "
        f"{result['relative_velocity'][0]:.12f} m/s"
    )

    print(
        f"  Final drag                     : "
        f"{result['drag']} N"
    )

    print(
        f"  Final net force                : "
        f"{result['net_force']} N"
    )


print()
print("=" * 80)
print("FRESH-PROCESS IMPORT")
print("=" * 80)

print(
    fresh_result.stdout.strip()
)

print(
    "[PASS] Public hydrolab3d.world API imports "
    "successfully in a fresh Python process."
)


print()
print("=" * 80)
print("FOCUSED CURRENT REGRESSION")
print("=" * 80)

print(
    pytest_result.stdout.strip()
)

print(
    "\n[PASS] Promoted ocean-current production implementation "
    "passed canonical validation and focused regression."
)

STEP 7 - PRODUCTION CURRENT RE-IMPORT AND VALIDATION

Production current module:
  src/hydrolab3d/world/currents.py
[PASS] Current functions explicitly re-imported from the promoted production module.

PRODUCTION PASSIVE-DRIFT VALIDATION
Current velocity                  : [ 0.5  -0.25  0.1 ] m/s
Final vehicle velocity            : [ 0.47228509 -0.23614255  0.09445702] m/s
Final relative velocity           : [-0.02771491  0.01385745 -0.00554298] m/s
Numerical final relative speed    : 0.031478083222 m/s
Analytical final relative speed   : 0.031485252084 m/s
Maximum relative-speed error      : 1.484482695453e-04 m/s

PRODUCTION THRUST/CURRENT VALIDATION

Zero current
  Current velocity               : [0. 0. 0.] m/s
  Expected terminal vx           : 2.000000000000 m/s
  Numerical terminal vx          : 1.999999992493 m/s
  Relative terminal vx           : 1.999999992493 m/s
  Final drag                     : [-19.99999985  -0.          -0.        ] N
  Final net force                : 

## Step 8 - Generate Persistent Validation Artifacts and Run the Complete HydroLab-3D Regression Suite

The production ocean-current implementation has now passed:

- notebook-local primitive validation;
- current-aware drag validation;
- passive-drift validation;
- analytical thrust/current equilibrium validation;
- production promotion;
- explicit production re-import;
- fresh-process import;
- focused automated regression.

The final implementation step has two responsibilities:

1. preserve a compact set of scientifically useful Notebook 06 artifacts;
2. confirm that the new world-current layer does not break any previously validated HydroLab-3D functionality.

No new physical model is introduced in this step.

---

### Canonical Artifact Scenario

The persistent dataset will use the three canonical propulsion cases already validated:

- zero current;
- following current;
- opposing current.

The common propulsion parameters are:

$$
m
=
20\text{ kg}
$$

$$
F_T
=
20\text{ N}
$$

$$
C_D
=
5
$$

which gives the analytical water-relative terminal speed:

$$
v_{rel,\infty}
=
\sqrt{
\frac{F_T}{C_D}
}
=
2.0\text{ m/s}
$$

The three current conditions are:

$$
v_{c,x}
=
0.0\text{ m/s}
$$

$$
v_{c,x}
=
+0.5\text{ m/s}
$$

and

$$
v_{c,x}
=
-0.5\text{ m/s}
$$

giving expected world-frame terminal velocities of:

$$
2.0\text{ m/s}
$$

$$
2.5\text{ m/s}
$$

and

$$
1.5\text{ m/s}
$$

respectively.

---

### Persistent Dataset

A single CSV will preserve the complete canonical histories for all three cases.

Each row will contain:

- case name;
- time;
- vehicle position;
- vehicle world velocity;
- current velocity;
- water-relative velocity;
- thrust force;
- drag force;
- net force;
- acceleration.

This provides enough information to reconstruct the physical force balance and verify the environmental coupling after the notebook is no longer running.

---

### Velocity Validation Plot

The first plot will compare world-frame longitudinal velocity for all three current cases.

It should visibly demonstrate:

$$
v_{x,\infty}
=
v_{c,x}
+
2.0
$$

The expected steady-state ordering is:

$$
v_{x,\infty}^{following}
>
v_{x,\infty}^{zero}
>
v_{x,\infty}^{opposing}
$$

with approximate values:

$$
2.5
>
2.0
>
1.5
\text{ m/s}
$$

---

### Water-Relative Velocity Plot

A second plot will show:

$$
v_{rel,x}
=
v_x-v_{c,x}
$$

for all three cases.

Although their world-frame velocities differ, all three curves should converge toward:

$$
\boxed{
v_{rel,x}
=
2.0\text{ m/s}
}
$$

This plot directly demonstrates the central physical result of Notebook 06:

> A uniform current shifts world-frame vehicle velocity without changing the steady thrust-drag equilibrium relative to the surrounding water.

---

### Force-Balance Plot

The final plot will use the following-current case and display:

- thrust force;
- drag force;
- net force.

At equilibrium:

$$
F_T
\rightarrow
20\text{ N}
$$

$$
F_D
\rightarrow
-20\text{ N}
$$

and therefore:

$$
F_{net}
\rightarrow
0
$$

This provides persistent evidence that the shifted world-frame terminal velocity is still produced by the same water-relative force balance.

---

### Output Directory

Notebook 06 owns:

`results/notebooks/06_ocean_currents/`

The intended artifact set is:

```text
results/notebooks/06_ocean_currents/
├── data/
│   └── current_validation.csv
└── plots/
    ├── velocity_validation.png
    ├── relative_velocity_validation.png
    └── force_balance.png
```

No decorative artifacts will be generated.

---

### Artifact Integrity Checks

After saving the files, the notebook will verify:

- every expected artifact exists;
- every file is non-empty;
- the CSV contains the expected columns;
- all stored numerical values are finite;
- all three current cases are represented;
- every case contains the expected number of samples;
- terminal velocities stored in the CSV reproduce the validated numerical results.

---

### Complete Repository Regression

Notebook 05 entered this notebook with:

`69 passed`

Notebook 06 has added a focused current regression suite containing:

`26 passed`

The total repository test count will **not** be predicted.

Instead, the complete suite will be executed and the actual result reported by `pytest` will become the Notebook 06 regression baseline.

The notebook is considered implementation-complete only if the entire HydroLab-3D test suite passes.

If this final step succeeds, no additional implementation is required and Notebook 06 can conclude with its engineering handoff to:

`07_six_dof_uuv.ipynb`

In [9]:
import csv
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# =====================================================================
# OUTPUT DIRECTORIES
# =====================================================================

output_root = (
    repo_root
    / "results"
    / "notebooks"
    / "06_ocean_currents"
)

data_dir = output_root / "data"
plots_dir = output_root / "plots"

data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

plots_dir.mkdir(
    parents=True,
    exist_ok=True,
)


csv_path = (
    data_dir
    / "current_validation.csv"
)

velocity_plot_path = (
    plots_dir
    / "velocity_validation.png"
)

relative_velocity_plot_path = (
    plots_dir
    / "relative_velocity_validation.png"
)

force_balance_plot_path = (
    plots_dir
    / "force_balance.png"
)


# =====================================================================
# REGENERATE CANONICAL HISTORIES THROUGH PRODUCTION CODE
# =====================================================================

artifact_dt = 0.01
artifact_duration = 20.0

artifact_num_steps = int(
    round(
        artifact_duration
        / artifact_dt
    )
)


artifact_mass = 20.0
artifact_drag_coefficient = 5.0

artifact_fluid_density = 1000.0
artifact_gravity = 9.81
artifact_volume = 0.020

artifact_max_thrust = 40.0
artifact_command = 0.5


artifact_gravity_force = gravity_force(
    artifact_mass,
    artifact_gravity,
)

artifact_buoyancy_force = buoyancy_force(
    artifact_fluid_density,
    artifact_volume,
    artifact_gravity,
)

artifact_hydrostatic_force = (
    artifact_gravity_force
    + artifact_buoyancy_force
)


np.testing.assert_allclose(
    artifact_hydrostatic_force,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


artifact_thrust = thruster_force_world(
    artifact_command,
    artifact_max_thrust,
    np.array([1.0, 0.0, 0.0]),
    0.0,
    0.0,
    0.0,
)


np.testing.assert_allclose(
    artifact_thrust,
    np.array([20.0, 0.0, 0.0]),
    rtol=0.0,
    atol=1e-12,
)


def generate_artifact_case(
    case_name,
    current_velocity,
):
    current_velocity = (
        production_uniform_current_velocity(
            current_velocity
        )
    )

    position = np.array(
        [0.0, 0.0, -10.0],
        dtype=float,
    )

    velocity = np.zeros(
        3,
        dtype=float,
    )

    records = []

    for step in range(
        artifact_num_steps + 1
    ):
        time = (
            step
            * artifact_dt
        )

        relative_velocity = (
            production_relative_water_velocity(
                velocity,
                current_velocity,
            )
        )

        drag_force = quadratic_drag_force(
            relative_velocity,
            artifact_drag_coefficient,
        )

        net_force = (
            artifact_thrust
            + drag_force
            + artifact_hydrostatic_force
        )

        acceleration = acceleration_from_force(
            net_force,
            artifact_mass,
        )

        records.append(
            {
                "case": case_name,
                "time": time,

                "position_x": position[0],
                "position_y": position[1],
                "position_z": position[2],

                "velocity_x": velocity[0],
                "velocity_y": velocity[1],
                "velocity_z": velocity[2],

                "current_x": current_velocity[0],
                "current_y": current_velocity[1],
                "current_z": current_velocity[2],

                "relative_velocity_x": relative_velocity[0],
                "relative_velocity_y": relative_velocity[1],
                "relative_velocity_z": relative_velocity[2],

                "thrust_x": artifact_thrust[0],
                "thrust_y": artifact_thrust[1],
                "thrust_z": artifact_thrust[2],

                "drag_x": drag_force[0],
                "drag_y": drag_force[1],
                "drag_z": drag_force[2],

                "net_force_x": net_force[0],
                "net_force_y": net_force[1],
                "net_force_z": net_force[2],

                "acceleration_x": acceleration[0],
                "acceleration_y": acceleration[1],
                "acceleration_z": acceleration[2],
            }
        )

        if step == artifact_num_steps:
            break

        position = integrate_position(
            position,
            velocity,
            artifact_dt,
        )

        velocity = integrate_velocity(
            velocity,
            acceleration,
            artifact_dt,
        )

    return records


artifact_case_specs = {
    "zero": np.array(
        [0.0, 0.0, 0.0],
        dtype=float,
    ),
    "following": np.array(
        [0.5, 0.0, 0.0],
        dtype=float,
    ),
    "opposing": np.array(
        [-0.5, 0.0, 0.0],
        dtype=float,
    ),
}


artifact_records = []

for case_name, current_velocity in artifact_case_specs.items():
    artifact_records.extend(
        generate_artifact_case(
            case_name,
            current_velocity,
        )
    )


# =====================================================================
# SAVE CSV
# =====================================================================

fieldnames = list(
    artifact_records[0].keys()
)


with csv_path.open(
    "w",
    newline="",
    encoding="utf-8",
) as csv_file:

    writer = csv.DictWriter(
        csv_file,
        fieldnames=fieldnames,
    )

    writer.writeheader()
    writer.writerows(
        artifact_records
    )


# =====================================================================
# CONVERT RECORDS TO NUMPY ARRAYS FOR PLOTTING
# =====================================================================

def case_records(case_name):
    return [
        record
        for record in artifact_records
        if record["case"] == case_name
    ]


def column_array(
    records,
    column,
):
    return np.asarray(
        [
            record[column]
            for record in records
        ],
        dtype=float,
    )


# =====================================================================
# PLOT 1 - WORLD-FRAME VELOCITY
# =====================================================================

plt.figure(
    figsize=(9, 5)
)

for case_name in [
    "zero",
    "following",
    "opposing",
]:
    records = case_records(
        case_name
    )

    plt.plot(
        column_array(
            records,
            "time",
        ),
        column_array(
            records,
            "velocity_x",
        ),
        label=case_name.capitalize(),
    )


plt.axhline(
    2.0,
    linestyle="--",
    linewidth=1.0,
    label="Zero-current terminal reference",
)

plt.axhline(
    2.5,
    linestyle=":",
    linewidth=1.0,
)

plt.axhline(
    1.5,
    linestyle=":",
    linewidth=1.0,
)

plt.xlabel(
    "Time [s]"
)

plt.ylabel(
    "World-frame x velocity [m/s]"
)

plt.title(
    "Ocean Current Shift of World-Frame Terminal Velocity"
)

plt.grid(
    True,
    alpha=0.3,
)

plt.legend()

plt.tight_layout()

plt.savefig(
    velocity_plot_path,
    dpi=150,
)

plt.close()


# =====================================================================
# PLOT 2 - WATER-RELATIVE VELOCITY
# =====================================================================

plt.figure(
    figsize=(9, 5)
)

for case_name in [
    "zero",
    "following",
    "opposing",
]:
    records = case_records(
        case_name
    )

    plt.plot(
        column_array(
            records,
            "time",
        ),
        column_array(
            records,
            "relative_velocity_x",
        ),
        label=case_name.capitalize(),
    )


plt.axhline(
    2.0,
    linestyle="--",
    linewidth=1.0,
    label="Analytical terminal relative velocity",
)

plt.xlabel(
    "Time [s]"
)

plt.ylabel(
    "Water-relative x velocity [m/s]"
)

plt.title(
    "Water-Relative Terminal Velocity Under Uniform Currents"
)

plt.grid(
    True,
    alpha=0.3,
)

plt.legend()

plt.tight_layout()

plt.savefig(
    relative_velocity_plot_path,
    dpi=150,
)

plt.close()


# =====================================================================
# PLOT 3 - FORCE BALANCE FOR FOLLOWING CURRENT
# =====================================================================

following_records = case_records(
    "following"
)

following_time = column_array(
    following_records,
    "time",
)

following_thrust_x = column_array(
    following_records,
    "thrust_x",
)

following_drag_x = column_array(
    following_records,
    "drag_x",
)

following_net_force_x = column_array(
    following_records,
    "net_force_x",
)


plt.figure(
    figsize=(9, 5)
)

plt.plot(
    following_time,
    following_thrust_x,
    label="Thrust",
)

plt.plot(
    following_time,
    following_drag_x,
    label="Drag",
)

plt.plot(
    following_time,
    following_net_force_x,
    label="Net force",
)

plt.xlabel(
    "Time [s]"
)

plt.ylabel(
    "x force [N]"
)

plt.title(
    "Following-Current Thrust-Drag Force Balance"
)

plt.grid(
    True,
    alpha=0.3,
)

plt.legend()

plt.tight_layout()

plt.savefig(
    force_balance_plot_path,
    dpi=150,
)

plt.close()


# =====================================================================
# ARTIFACT INTEGRITY VALIDATION
# =====================================================================

expected_artifacts = [
    csv_path,
    velocity_plot_path,
    relative_velocity_plot_path,
    force_balance_plot_path,
]


for artifact_path in expected_artifacts:

    assert artifact_path.is_file(), (
        f"Missing artifact: {artifact_path}"
    )

    assert artifact_path.stat().st_size > 0, (
        f"Empty artifact: {artifact_path}"
    )


expected_samples_per_case = (
    artifact_num_steps
    + 1
)

expected_total_rows = (
    expected_samples_per_case
    * len(artifact_case_specs)
)


assert (
    len(artifact_records)
    == expected_total_rows
)


for case_name in artifact_case_specs:

    records = case_records(
        case_name
    )

    assert (
        len(records)
        == expected_samples_per_case
    )


# =====================================================================
# FINITE CSV DATA VALIDATION
# =====================================================================

numeric_fieldnames = [
    field
    for field in fieldnames
    if field != "case"
]


for record in artifact_records:

    numeric_values = np.asarray(
        [
            record[field]
            for field in numeric_fieldnames
        ],
        dtype=float,
    )

    assert np.all(
        np.isfinite(
            numeric_values
        )
    )


# =====================================================================
# TERMINAL VALUES RECOVERED FROM SAVED DATA
# =====================================================================

expected_terminal_vx = {
    "zero": 2.0,
    "following": 2.5,
    "opposing": 1.5,
}


artifact_terminal_results = {}

for case_name in artifact_case_specs:

    records = case_records(
        case_name
    )

    final_record = records[-1]

    final_world_vx = (
        final_record[
            "velocity_x"
        ]
    )

    final_relative_vx = (
        final_record[
            "relative_velocity_x"
        ]
    )

    artifact_terminal_results[
        case_name
    ] = {
        "world_vx": final_world_vx,
        "relative_vx": final_relative_vx,
    }

    np.testing.assert_allclose(
        final_world_vx,
        expected_terminal_vx[
            case_name
        ],
        rtol=0.0,
        atol=1e-6,
    )

    np.testing.assert_allclose(
        final_relative_vx,
        2.0,
        rtol=0.0,
        atol=1e-6,
    )


# =====================================================================
# COMPLETE HYDROLAB-3D REGRESSION SUITE
# =====================================================================

full_test_env = os.environ.copy()

existing_pythonpath = full_test_env.get(
    "PYTHONPATH",
    "",
)

full_test_env["PYTHONPATH"] = (
    str(src_path)
    if not existing_pythonpath
    else (
        str(src_path)
        + os.pathsep
        + existing_pythonpath
    )
)


full_pytest_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
    ],
    cwd=repo_root,
    env=full_test_env,
    capture_output=True,
    text=True,
)


if full_pytest_result.returncode != 0:

    print(
        full_pytest_result.stdout
    )

    print(
        full_pytest_result.stderr
    )

    raise RuntimeError(
        "Complete HydroLab-3D regression suite failed."
    )


# =====================================================================
# FINAL STEP SUMMARY
# =====================================================================

print("=" * 80)
print("STEP 8 - NOTEBOOK 06 ARTIFACT AND REGRESSION VALIDATION")
print("=" * 80)


print("\nPersistent dataset")
print(
    f"  CSV rows                       : "
    f"{len(artifact_records)}"
)

print(
    f"  Samples per current case       : "
    f"{expected_samples_per_case}"
)

print(
    f"  Cases                          : "
    f"{', '.join(artifact_case_specs.keys())}"
)


print("\nSaved terminal values")

for case_name in [
    "zero",
    "following",
    "opposing",
]:

    result = artifact_terminal_results[
        case_name
    ]

    print(
        f"  {case_name.capitalize():9s}"
        f" world vx = "
        f"{result['world_vx']:.12f} m/s"
        f" | relative vx = "
        f"{result['relative_vx']:.12f} m/s"
    )


print("\nGenerated artifacts")

for artifact_path in expected_artifacts:

    print(
        f"  "
        f"{artifact_path.relative_to(repo_root)}"
        f"  "
        f"({artifact_path.stat().st_size} bytes)"
    )


print()
print("=" * 80)
print("COMPLETE HYDROLAB-3D REGRESSION")
print("=" * 80)

print(
    full_pytest_result.stdout.strip()
)


print(
    "\n[PASS] Notebook 06 persistent artifacts validated."
)

print(
    "[PASS] Complete HydroLab-3D regression suite passed."
)

print(
    "[PASS] Notebook 06 implementation is complete and "
    "ready for engineering handoff."
)

STEP 8 - NOTEBOOK 06 ARTIFACT AND REGRESSION VALIDATION

Persistent dataset
  CSV rows                       : 6003
  Samples per current case       : 2001
  Cases                          : zero, following, opposing

Saved terminal values
  Zero      world vx = 1.999999992493 m/s | relative vx = 1.999999992493 m/s
  Following world vx = 2.499999987720 m/s | relative vx = 1.999999987720 m/s
  Opposing  world vx = 1.499999995506 m/s | relative vx = 1.999999995506 m/s

Generated artifacts
  results/notebooks/06_ocean_currents/data/current_validation.csv  (1268905 bytes)
  results/notebooks/06_ocean_currents/plots/velocity_validation.png  (77500 bytes)
  results/notebooks/06_ocean_currents/plots/relative_velocity_validation.png  (74837 bytes)
  results/notebooks/06_ocean_currents/plots/force_balance.png  (52158 bytes)

COMPLETE HYDROLAB-3D REGRESSION
........................................................................ [ 75%]
.......................                                     

# Conclusion

## What Was Implemented

Notebook 06 introduced the first explicit **environmental-flow model** into HydroLab-3D.

The validated reduced translational model now accounts for ocean current through the vehicle's velocity relative to the surrounding water:

$$
\boxed{
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
}
$$

where:

- $\mathbf{v}$ is vehicle velocity in the HydroLab-3D world frame;
- $\mathbf{v}_c$ is ocean-current velocity in the same world frame;
- $\mathbf{v}_{rel}$ is vehicle velocity relative to the water.

The existing quadratic hydrodynamic-drag model was preserved and composed with the new relative-velocity layer:

$$
\boxed{
\mathbf{F}_D
=
-C_D
\|\mathbf{v}_{rel}\|
\mathbf{v}_{rel}
}
$$

or equivalently:

$$
\boxed{
\mathbf{F}_D
=
-C_D
\|\mathbf{v}-\mathbf{v}_c\|
(\mathbf{v}-\mathbf{v}_c)
}
$$

The complete reduced translational force balance entering the next stage of HydroLab-3D is therefore:

$$
\boxed{
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D(\mathbf{v}-\mathbf{v}_c)
+
\mathbf{F}_G
+
\mathbf{F}_B
}
$$

The current model was intentionally kept minimal.

Notebook 06 implements:

- uniform world-frame current velocity;
- strict finite 3-vector validation;
- water-relative vehicle velocity;
- current-aware drag through composition with the existing drag model;
- passive drift under uniform current;
- propulsion under zero, following, and opposing currents;
- analytical current/thrust equilibrium validation;
- persistent validation artifacts;
- production promotion;
- focused automated regression;
- complete repository regression.

No CFD or higher-order ocean dynamics were introduced.

---

## What Was Validated

### Uniform Current Representation

The current model correctly represents constant world-frame current vectors of the form:

$$
\mathbf{v}_c
=
\begin{bmatrix}
v_{c,x}\\
v_{c,y}\\
v_{c,z}
\end{bmatrix}
$$

Validation confirmed:

- exact zero-current representation;
- exact known 3D current representation;
- finite-value enforcement;
- exact `(3,)` shape enforcement;
- independent returned storage rather than aliasing caller-owned arrays.

---

### Water-Relative Velocity

The implementation correctly evaluates:

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

For the test case:

$$
\mathbf{v}
=
\begin{bmatrix}
2.0\\
-1.0\\
0.5
\end{bmatrix}
\mathrm{m/s}
$$

and:

$$
\mathbf{v}_c
=
\begin{bmatrix}
0.5\\
0.25\\
-0.1
\end{bmatrix}
\mathrm{m/s}
$$

the expected and computed result was:

$$
\mathbf{v}_{rel}
=
\begin{bmatrix}
1.5\\
-1.25\\
0.6
\end{bmatrix}
\mathrm{m/s}
$$

A stationary vehicle correctly produced:

$$
\mathbf{v}_{rel}
=
-\mathbf{v}_c
$$

while a vehicle moving exactly with the surrounding water produced:

$$
\mathbf{v}_{rel}
=
\mathbf{0}
$$

---

### Current-Aware Quadratic Drag

Notebook 06 demonstrated that the validated Notebook 03 drag implementation did not need to be rewritten.

Instead:

```text
vehicle world velocity
        +
ocean current velocity
        ↓
water-relative velocity
        ↓
quadratic_drag_force()
        ↓
hydrodynamic drag
```

For a stationary vehicle in the test current:

$$
\mathbf{v}_c
=
\begin{bmatrix}
0.5\\
0.25\\
-0.1
\end{bmatrix}
\mathrm{m/s}
$$

the vehicle had:

$$
\mathbf{v}_{rel}
=
\begin{bmatrix}
-0.5\\
-0.25\\
0.1
\end{bmatrix}
\mathrm{m/s}
$$

and the resulting drag was:

$$
\mathbf{F}_D
=
\begin{bmatrix}
0.85183625\\
0.42591813\\
-0.17036725
\end{bmatrix}
\mathrm{N}
$$

The positive current-alignment test confirmed that this force acts in the physically correct direction.

For the arbitrary 3D case:

$$
\mathbf{v}_{rel}
=
\begin{bmatrix}
1.5\\
-1.25\\
0.6
\end{bmatrix}
\mathrm{m/s}
$$

the calculated drag was:

$$
\mathbf{F}_D
=
\begin{bmatrix}
-9.19201420\\
7.66001183\\
-3.67680568
\end{bmatrix}
\mathrm{N}
$$

with:

$$
\mathbf{F}_D
\cdot
\mathbf{v}_{rel}
=
-25.569119491647
$$

confirming dissipative behavior relative to the surrounding fluid.

Zero current reproduced the original drag implementation exactly.

---

## Passive Current Drift

A passive neutrally buoyant vehicle was initialized at rest inside the uniform current:

$$
\mathbf{v}_c
=
\begin{bmatrix}
0.50\\
-0.25\\
0.10
\end{bmatrix}
\mathrm{m/s}
$$

using:

$$
m
=
20\text{ kg}
$$

$$
\rho
=
1000\text{ kg/m}^3
$$

$$
V
=
0.020\text{ m}^3
$$

$$
C_D
=
5
$$

Gravity and buoyancy cancelled exactly:

$$
\mathbf{F}_G
=
\begin{bmatrix}
0\\
0\\
-196.2
\end{bmatrix}
\mathrm{N}
$$

$$
\mathbf{F}_B
=
\begin{bmatrix}
0\\
0\\
196.2
\end{bmatrix}
\mathrm{N}
$$

therefore:

$$
\mathbf{F}_G+\mathbf{F}_B
=
\mathbf{0}
$$

The initial relative speed was:

$$
\|\mathbf{v}_{rel}(0)\|
=
0.567890834580\text{ m/s}
$$

After $120$ s, the vehicle velocity became:

$$
\mathbf{v}
=
\begin{bmatrix}
0.47228509\\
-0.23614255\\
0.09445702
\end{bmatrix}
\mathrm{m/s}
$$

compared with:

$$
\mathbf{v}_c
=
\begin{bmatrix}
0.5\\
-0.25\\
0.1
\end{bmatrix}
\mathrm{m/s}
$$

The remaining relative velocity was:

$$
\mathbf{v}_{rel}
=
\begin{bmatrix}
-0.02771491\\
0.01385745\\
-0.00554298
\end{bmatrix}
\mathrm{m/s}
$$

with relative speed:

$$
\|\mathbf{v}_{rel}\|
=
0.031478083222\text{ m/s}
$$

The vehicle therefore moved strongly toward the surrounding water velocity, as required.

---

## Analytical Passive-Drift Validation

For constant uniform current and quadratic drag, the relative-speed dynamics reduce to:

$$
m\dot{s}
=
-C_Ds^2
$$

with analytical solution:

$$
\boxed{
s(t)
=
\frac{s_0}{
1+
\frac{C_D}{m}s_0t
}
}
$$

At the final simulation time, the analytical relative speed was:

$$
s_{analytical}
=
0.031485252084\text{ m/s}
$$

while the numerical result was:

$$
s_{numerical}
=
0.031478083222\text{ m/s}
$$

The maximum absolute relative-speed error over the complete trajectory was:

$$
\boxed{
1.484482695453\times10^{-4}\text{ m/s}
}
$$

This provides quantitative confirmation that the passive current-drift dynamics agree with the analytical quadratic-drag solution.

---

## Thrust Under Uniform Current

The canonical Notebook 05 propulsion configuration was retained:

$$
F_{max}
=
40\text{ N}
$$

$$
u
=
0.5
$$

giving:

$$
\mathbf{F}_T
=
\begin{bmatrix}
20\\
0\\
0
\end{bmatrix}
\mathrm{N}
$$

For:

$$
C_D
=
5
$$

the analytical water-relative terminal speed is:

$$
v_{rel,\infty}
=
\sqrt{
\frac{20}{5}
}
=
2.0\text{ m/s}
$$

Three current cases were evaluated.

### Zero Current

Expected:

$$
v_{x,\infty}
=
2.0\text{ m/s}
$$

Numerical:

$$
\boxed{
v_{x,\infty}
=
1.999999992493\text{ m/s}
}
$$

Water-relative terminal velocity:

$$
1.999999992493\text{ m/s}
$$

---

### Following Current

For:

$$
v_{c,x}
=
+0.5\text{ m/s}
$$

the analytical prediction is:

$$
v_{x,\infty}
=
0.5+2.0
=
2.5\text{ m/s}
$$

Numerical:

$$
\boxed{
v_{x,\infty}
=
2.499999987720\text{ m/s}
}
$$

while:

$$
v_{rel,x,\infty}
=
1.999999987720\text{ m/s}
$$

The terminal drag was:

$$
\mathbf{F}_D
\approx
\begin{bmatrix}
-19.99999975\\
0\\
0
\end{bmatrix}
\mathrm{N}
$$

and the remaining net force was only:

$$
\mathbf{F}_{net}
\approx
\begin{bmatrix}
2.48086991\times10^{-7}\\
0\\
0
\end{bmatrix}
\mathrm{N}
$$

---

### Opposing Current

For:

$$
v_{c,x}
=
-0.5\text{ m/s}
$$

the analytical prediction is:

$$
v_{x,\infty}
=
-0.5+2.0
=
1.5\text{ m/s}
$$

Numerical:

$$
\boxed{
v_{x,\infty}
=
1.499999995506\text{ m/s}
}
$$

while:

$$
v_{rel,x,\infty}
=
1.999999995506\text{ m/s}
$$

The terminal drag was:

$$
\mathbf{F}_D
\approx
\begin{bmatrix}
-19.99999991\\
0\\
0
\end{bmatrix}
\mathrm{N}
$$

with terminal net force approximately zero.

---

## Key Results

The central numerical result of Notebook 06 is:

| Current case | $v_{c,x}$ [m/s] | Numerical world terminal $v_x$ [m/s] | Numerical water-relative terminal $v_{rel,x}$ [m/s] |
|---|---:|---:|---:|
| Zero | $0.0$ | $1.999999992493$ | $1.999999992493$ |
| Following | $+0.5$ | $2.499999987720$ | $1.999999987720$ |
| Opposing | $-0.5$ | $1.499999995506$ | $1.999999995506$ |

The measured world-frame shifts relative to the zero-current case were:

$$
\Delta v_{following}
=
0.499999995227\text{ m/s}
$$

and:

$$
\Delta v_{opposing}
=
-0.499999996987\text{ m/s}
$$

These values agree with the imposed currents to numerical precision.

The result establishes the fundamental current-model invariant:

$$
\boxed{
\text{uniform current shifts world-frame terminal velocity}
}
$$

while:

$$
\boxed{
\text{water-relative thrust-drag equilibrium remains unchanged}
}
$$

---

## Production-Code Promotion

The validated environmental-flow primitives were promoted into:

`src/hydrolab3d/world/currents.py`

with:

- `uniform_current_velocity()`
- `relative_water_velocity()`

They are publicly exported through:

`src/hydrolab3d/world/__init__.py`

The existing drag implementation in:

`src/hydrolab3d/dynamics/drag.py`

was intentionally left unchanged.

The resulting production architecture is:

```text
World / Environment
        │
        ▼
uniform_current_velocity()
        │
        ▼
relative_water_velocity()
        │
        ▼
quadratic_drag_force()
        │
        ▼
Reduced Vehicle Dynamics
```

This maintains a clean separation between environmental state and vehicle hydrodynamics.

After production promotion, the implementation was explicitly re-imported from:

`src/hydrolab3d/world/currents.py`

and the canonical passive-drift and thrust/current experiments were rerun through the production implementation.

The production results reproduced the notebook prototypes.

A fresh Python process also successfully imported:

`hydrolab3d.world`

and evaluated both public current functions.

---

## Automated Regression

A focused current regression suite was implemented in:

`tests/test_currents.py`

The focused result was:

```text
26 passed
```

The complete HydroLab-3D regression suite was then executed.

Notebook 05 entered this notebook with a baseline of:

```text
69 passed
```

After integrating the ocean-current layer, the complete repository produced:

```text
95 passed in 0.14s
```

Therefore:

$$
\boxed{
95\text{ tests passed}
}
$$

with no regression in the previously validated HydroLab-3D stack.

---

## Generated Outputs

Notebook 06 generated the following persistent artifacts:

### Data

`results/notebooks/06_ocean_currents/data/current_validation.csv`

The canonical dataset contains:

```text
6003 rows
```

consisting of:

```text
2001 samples × 3 current cases
```

for:

- `zero`
- `following`
- `opposing`

The CSV records:

- time;
- position;
- vehicle world velocity;
- ocean-current velocity;
- water-relative velocity;
- thrust;
- drag;
- net force;
- acceleration.

---

### Plots

`results/notebooks/06_ocean_currents/plots/velocity_validation.png`

Shows the shift in world-frame terminal velocity between zero, following, and opposing current cases.

`results/notebooks/06_ocean_currents/plots/relative_velocity_validation.png`

Shows all three cases converging toward the same approximately:

$$
2.0\text{ m/s}
$$

water-relative terminal velocity.

`results/notebooks/06_ocean_currents/plots/force_balance.png`

Shows the following-current case converging toward:

$$
F_T
=
20\text{ N}
$$

$$
F_D
=
-20\text{ N}
$$

and:

$$
F_{net}
=
0
$$

---

## Output Location

All Notebook 06 persistent outputs are stored under:

`results/notebooks/06_ocean_currents/`

with the final structure:

```text
results/notebooks/06_ocean_currents/
├── data/
│   └── current_validation.csv
└── plots/
    ├── velocity_validation.png
    ├── relative_velocity_validation.png
    └── force_balance.png
```

All artifacts were verified to exist and contain non-empty data.

All stored numerical histories were verified to remain finite.

---

## Important Observations

### 1. Current Must Enter Through Relative Water Velocity

The defining modeling decision of Notebook 06 is:

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

The current is not independently added to position while ordinary world-relative drag is also applied.

Doing so would double-count environmental transport.

Instead:

```text
current
   ↓
relative velocity
   ↓
drag
   ↓
acceleration
   ↓
vehicle world velocity
   ↓
position
```

The resulting passive-drift behavior emerges dynamically from the force model.

---

### 2. The Existing Drag Model Required No Modification

Ocean-current support did not require rewriting:

`quadratic_drag_force()`

The same validated quadratic-drag function accepts the water-relative velocity as its input.

This preserves the Notebook 03 physics and keeps HydroLab-3D modular.

---

### 3. Ground-Relative and Water-Relative Velocity Are Different Physical Quantities

Under constant propulsion:

$$
v_{rel,\infty}
=
\sqrt{
\frac{F_T}{C_D}
}
$$

while:

$$
v_{\infty}
=
v_c
+
v_{rel,\infty}
$$

Thus a following current increases world-frame speed and an opposing current decreases it, even though the equilibrium speed through the water remains unchanged.

This distinction will become increasingly important for:

- navigation;
- station keeping;
- DVL modeling;
- trajectory control;
- energy-aware planning;
- current compensation.

---

### 4. Passive Drift Is Asymptotic

Under quadratic drag alone, a passive vehicle approaches the surrounding current velocity continuously.

The water-relative speed obeys:

$$
s(t)
=
\frac{s_0}{
1+
\frac{C_D}{m}s_0t
}
$$

so exact equality:

$$
\mathbf{v}
=
\mathbf{v}_c
$$

is approached asymptotically rather than reached at a finite simulation time.

The numerical results correctly reproduce this behavior.

---

### 5. Uniform Current Is Sufficient for the Foundational Validation

Notebook 06 did not need:

- depth gradients;
- oscillatory currents;
- vortices;
- spatial current maps;
- stochastic turbulence.

A constant 3D current was sufficient to establish the architecture and validate the physically important relationship between environmental flow, relative motion, drag, and vehicle dynamics.

More sophisticated current fields can later reuse the same interface without changing the fundamental coupling.

---

## Current Limitations

The current environmental-flow model remains intentionally reduced.

Notebook 06 does not model:

- computational fluid dynamics;
- Navier-Stokes flow;
- turbulence;
- wave-induced motion;
- vortex shedding;
- spatially varying currents;
- depth-dependent currents;
- time-varying currents;
- current-field interpolation;
- terrain-flow interaction;
- boundary layers;
- rotational hydrodynamics;
- added mass;
- Coriolis or centripetal effects;
- six-degree-of-freedom marine dynamics.

The drag coefficient remains the previously validated simplified scalar coefficient rather than a full body-axis damping model.

These limitations are consistent with the purpose of Notebook 06: establish trustworthy current-relative translational physics before increasing model fidelity.

---

## How This Result Will Be Used

Notebook 06 completes the environmental component required by the reduced HydroLab-3D translational stack.

The simulator now has validated foundations for:

- 3D position integration;
- orientation and body/world transformations;
- force-to-acceleration conversion;
- velocity integration;
- quadratic drag;
- gravity;
- buoyancy;
- normalized thruster commands;
- thrust saturation;
- bidirectional thrust;
- thruster direction;
- body-to-world thrust transformation;
- world-frame ocean-current representation;
- water-relative velocity;
- current-aware quadratic drag;
- passive current drift;
- propulsion under following and opposing current.

The validated reduced force model is:

$$
\boxed{
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D(\mathbf{v}-\mathbf{v}_c)
+
\mathbf{F}_G
+
\mathbf{F}_B
}
$$

This model is now sufficiently validated to serve as the translational reference when HydroLab-3D advances to a complete marine vehicle formulation.

---

## Engineering Handoff

Notebook 07 may assume that the following have already been independently validated:

```text
3D kinematics
        +
orientation transforms
        +
force-to-acceleration dynamics
        +
velocity integration
        +
quadratic hydrodynamic drag
        +
gravity
        +
buoyancy
        +
thruster command mapping
        +
thruster saturation
        +
thruster direction
        +
body-to-world thrust transformation
        +
world-frame ocean current
        +
water-relative velocity
        +
current-aware drag
        +
passive current drift
```

These foundational components should not be reopened unless integration into the higher-fidelity model exposes a genuine inconsistency.

---

## Next Notebook

The next HydroLab-3D development notebook is:

`07_six_dof_uuv.ipynb`

Notebook 07 will begin the transition from the validated reduced translational model:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_T
+
\mathbf{F}_D(\mathbf{v}-\mathbf{v}_c)
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

toward the classical marine six-degree-of-freedom formulation:

$$
\boxed{
M\dot{\nu}
+
C(\nu)\nu
+
D(\nu)\nu
+
g(\eta)
=
\tau
+
\tau_{env}
}
$$

with vehicle pose evolution of the form:

$$
\dot{\eta}
=
J(\eta)\nu
$$

Notebook 07 can therefore focus on the genuinely new physics:

- six-degree-of-freedom state representation;
- rigid-body inertia;
- added mass;
- Coriolis and centripetal terms;
- rotational dynamics;
- marine damping structure;
- restoring forces and moments;
- actuator forces and moments;
- environmental coupling within the full marine model.

It does **not** need to rediscover whether kinematics, orientation, drag, hydrostatics, thrusters, or ocean-current-relative translational behavior work.

Those foundations are now validated.

$$
\boxed{
\text{Notebook 06 complete: the vehicle now moves through water, not merely through coordinates.}
}
$$

---